# Topic 04 · Reinforcement learning from human feedback (RLHF): reward models, PPO and DPO on lục bát verse

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trungnguyenvn/hoc-sau-notebooks/blob/main/cd04-rlhf-en.ipynb)

This notebook accompanies [Topic 04 of Học Sâu · Reels](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/rlhf/), a deep-learning course originally in Vietnamese.
It is the English edition of the course's notebook
([Vietnamese original](https://github.com/trungnguyenvn/hoc-sau-notebooks/blob/main/cd04-rlhf.ipynb)):
the code, the data and every check are the same.

Lesson 26 post-trained the number-reading model with a hand-written grader. A real chatbot has no such grader: all
there is are the times a rater **compared** two answers. This topic goes through the whole RLHF pipeline that
InstructGPT used, at a size that runs on a CPU: a small GPT writes the eight-syllable line that follows a
six-syllable line of *The Tale of Kiều* (*Truyện Kiều*, Nguyễn Du's Vietnamese poem in lục bát verse), a simulated
rater picks the better line of each pair, a reward model learns from those choices, then PPO optimizes the model
against that reward. DPO, for its part, skips the reward model. The rater's "taste" here is a program (the rules of
lục bát), so we can measure what real RLHF can't: the *true* quality of every line of verse, and the moment the
optimization starts going the wrong way. The verse is the models' data, so it stays Vietnamese; section 5 explains
the rules, and the text glosses the lines it quotes.

**What you'll do:**
1. Fit the Bradley–Terry model: from comparisons to scores, and what the scores cannot tell you.
2. Train a reward model from 1,000 comparisons, then compare it with the accuracy ceiling that a noisy rater sets.
3. See Goodhart's law: on an exact table, then with best-of-N and PPO on lục bát verse, and read the lines the model
   writes to please the reward model.
4. Assemble PPO yourself piece by piece (a per-token KL penalty, GAE, the clipped ratio) and see why putting the KL
   into the loss the wrong way means it no longer pulls the model back.
5. Run DPO and IPO: where they head when the rater never changes its mind, and why the probability of the *chosen*
   line can also drop.

**Background:** [Lesson 26 · From language model to assistant](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/hau-huan-luyen/)
(π*, DPO on a table, best-of-N), [Lesson 17 · Transformer & GPT](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/transformer/)
and [Topic 03](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/nghien-cuu-sinh-va-hau-huan-luyen/) (how SFT and
reinforcement learning differ in what they forget).
**Run time:** about 1 minute on a CPU, including the first data download.

The notebook uses **PyTorch**. Each section runs a short version live, then loads the full results (3 seeds, 200 PPO
iterations, 6 DPO passes) that the course's script `reel/rlhf/sweep.py` computed in advance, with the same code and
the same seeds.

> The **🤔 Predict first** cells ask a question before you run something. Answer it yourself, then run the next cell.

## 1 · Setup

In [ ]:
import base64
import copy
import hashlib
import json
import math
import os
import re
import unicodedata
import urllib.request
from itertools import pairwise
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "2")
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from matplotlib.ticker import FixedLocator, FuncFormatter, NullFormatter, NullLocator, ScalarFormatter
from torch import nn

torch.set_num_threads(2)

BLUE, ORANGE, GREEN, GREY = "#3987e5", "#d95926", "#199e70", "#8a94a6"
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})


def num(x, digits=2):
    """A number with thousands separators: 13,007 · 0.1 · −4.38 (never “−0”)."""
    return f"{round(float(x), digits) + 0.0:,.{digits}f}".replace("-", "−")


def pct(p, digits=1):
    return num(100 * p, digits) + "%"


SUP = str.maketrans("-0123456789", "⁻⁰¹²³⁴⁵⁶⁷⁸⁹")


def tick(v, _pos=None):
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:.4f}".rstrip("0").rstrip(".")
    return num(v, len(s.split(".")[1]) if "." in s else 0)


def log_tick(v, _pos=None):
    if 0.01 <= v <= 1000:
        return tick(v)
    e = math.floor(math.log10(v) + 1e-9)
    m = v / 10.0**e
    return (f"{tick(m)}·" if abs(m - 1) > 1e-9 else "") + f"10{str(e).translate(SUP)}"


def fmt_axes(fig):
    """Axis labels in the notebook's number format; call right before plt.show(). Short log axes get 1–2–5 ticks,
    long ones powers of 10."""
    for ax in fig.axes:
        for axis, lim in ((ax.xaxis, ax.get_xlim()), (ax.yaxis, ax.get_ylim())):
            if axis.get_scale() != "log":
                if isinstance(axis.get_major_formatter(), ScalarFormatter):
                    axis.set_major_formatter(FuncFormatter(tick))
                continue
            lo, hi = sorted(lim)
            k0, k1 = math.ceil(math.log10(lo) - 1e-9), math.floor(math.log10(hi) + 1e-9)
            if hi / lo < 300:
                ticks = [m * 10.0**k for k in range(k0 - 1, k1 + 1) for m in (1, 2, 5) if lo <= m * 10.0**k <= hi]
            else:
                step = max(1, math.ceil((k1 - k0) / 8))
                ticks = [10.0**k for k in range(k0, k1 + 1) if k % step == 0]
            axis.set_major_locator(FixedLocator(ticks))
            axis.set_minor_locator(NullLocator())
            axis.set_minor_formatter(NullFormatter())
            axis.set_major_formatter(FuncFormatter(log_tick))

## 2 · Data

*The Tale of Kiều* from the Vietnamese Wikisource, pinned at revision 205132 as in Lessons 15 and 17 (with an MD5
check), and two files from the script: `rlhf-lucbat.json` holds the results of every run, `rlhf-weights.json` the
float16 weights of three models (the SFT model, the reward model and one PPO policy) along with 2,500 pairs of lines
that the simulated rater has compared.

In [ ]:
KIEU_SRC = "https://vi.wikisource.org/w/index.php?oldid=205132&action=raw"
KIEU_RAW = Path("truyen_kieu_205132.wiki")
opener = urllib.request.build_opener()
opener.addheaders = [("User-Agent", "HocSauReels-notebook/1.0 (education; https://d3qr1cjln6qqtk.cloudfront.net/)")]
urllib.request.install_opener(opener)  # Wikimedia rejects requests that have no User-Agent
if not KIEU_RAW.exists():
    urllib.request.urlretrieve(KIEU_SRC, KIEU_RAW)
assert hashlib.md5(KIEU_RAW.read_bytes()).hexdigest() == "d4090daac094841888ee2b2b18661065"


def fetch_json(name, md5):
    p = Path("data") / name
    if not p.exists():
        p.parent.mkdir(exist_ok=True)
        urllib.request.urlretrieve("https://raw.githubusercontent.com/trungnguyenvn/hoc-sau-notebooks/main/data/" + name, p)
    assert hashlib.md5(p.read_bytes()).hexdigest() == md5, f"{name} is corrupted; delete it and download it again"
    return json.loads(p.read_text())


RUNS = fetch_json("rlhf-lucbat.json", "2ba3eaad71495e13bd29c03d0fe4907d")
W = fetch_json("rlhf-weights.json", "5be6aa64330bcdba4008ddeebbe0450d")

Each 6–8 couplet of the poem is one example: the **prompt** is the six-syllable line, the **response** is the
eight-syllable line right after it. The unit (token) is the syllable, lowercased, with punctuation removed
(Vietnamese writes each syllable as a separate word, so the syllables of a line are the words between its spaces). We
split by position in the poem: the first 1,300 couplets for training, the next 163 for validation, the last 164 for
testing. The cell prints the poem's first two couplets, roughly: “A hundred years in this world of ours, / talent and
destiny are apt to be at odds. / Having lived through an upheaval of seas turned into mulberry fields, / the things
one has seen wrench the heart.”

In [ ]:
body = KIEU_RAW.read_bytes().decode("utf-8").split("<poem>")[1].split("</poem>")[0]
raw = [re.sub(r"\{\{số\|\d+\}\}", "", ln).strip() for ln in body.split("\n")]
text = "\n".join(ln for ln in raw if ln) + "\n"
assert hashlib.md5(text.encode()).hexdigest() == "dd94ee390bfc543287b11af62b2f2fc8"
lines = [re.sub(r"[^\w\s]", " ", ln).lower().split() for ln in text[:-1].split("\n")]
couplets = [(a, b) for a, b in pairwise(lines) if len(a) == 6 and len(b) == 8]
PAD, SEP, EOS = 0, 1, 2
VOCAB = ["<pad>", "<sep>", "<eos>", *sorted({s for ln in lines for s in ln})]
IX = {s: i for i, s in enumerate(VOCAB)}
V = len(VOCAB)
X = torch.tensor([[IX[s] for s in a] + [SEP] + [IX[s] for s in b] + [EOS] for a, b in couplets])
X_train, X_val, X_test = X[:1300], X[1300:1463], X[1463:]
PLEN, RESP = 7, 13  # prompt: 6 syllables + <sep>; response: at most 13 tokens (syllables, then <eos> if any)
T_MAX = PLEN + RESP
print(
    f"{num(len(lines), 0)} lines of verse, {num(len(couplets), 0)} 6–8 couplets, "
    f"{num(V, 0)} tokens (including <pad>, <sep>, <eos>)"
)
print(f"training {num(len(X_train), 0)} · validation {len(X_val)} · test {len(X_test)} couplets")
for a, b in couplets[:2]:
    print("  ", " ".join(a), "/", " ".join(b))
assert (len(lines), len(couplets), V) == (3254, 1627, 2385)

## 3 · RLHF from 2017 to today

The pipeline this topic rebuilds has three steps (Ouyang et al., 2022, known as InstructGPT):

1. **SFT.** Fine-tune the pretrained model on example responses. The result is $\pi_{\text{ref}}$.
2. **Reward model.** Have $\pi_{\text{ref}}$ write several responses, have raters rank them, then train a model
   $r_\phi(x, y)$ to predict which response is chosen.
3. **Reinforcement learning.** Optimize $\pi$ for a high $r_\phi$ without going too far from $\pi_{\text{ref}}$:
   $\max_\pi \; \mathbb{E}_{y\sim\pi}[r_\phi(x, y)] - \beta\,\mathrm{KL}(\pi\,\|\,\pi_{\text{ref}})$, with PPO.

A few milestones:
- **Christiano et al. (2017)** taught agents to play Atari and to control simulated robots only from the times a
  human watcher compared two short video clips. This is where the reward model learned from comparisons was born.
- **Ziegler et al. (2019), Stiennon et al. (2020)** brought that approach to language models (continuing text,
  summarization), with a per-token KL penalty.
- **InstructGPT (2022)**: about 13 thousand prompts for SFT, 33 thousand for the reward model, 31 thousand for PPO;
  raters ranked 4 to 9 responses at a time; a reward model of 6 billion parameters. Raters preferred the responses of
  the 1.3-billion-parameter model after RLHF to those of the 175-billion-parameter GPT-3. Two raters agreed with each
  other only 72.6–77.3% of the time, and the reward model predicted their choices correctly 69.6–72.4% of the time:
  comparison data is inherently very noisy.
- **Bai et al. (2022)** trained a "helpful and harmless" assistant with the same pipeline; *Constitutional AI*
  replaces part of the raters with the model itself, grading by a set of principles (RLAIF).
- **DPO (Rafailov et al., 2023)** drops the reward model altogether and learns directly from the pairs (section 10).
- **Llama 3 (2024)** iterates several rounds: reward model → rejection sampling (best-of-N) → SFT → DPO. **Tülu 3
  (2024)**: SFT → DPO → reinforcement learning with verifiable rewards (RLVR).
- **DeepSeek-R1 (2025)** uses GRPO with a reward made of rules that check the answer; during reinforcement learning,
  the pass@1 score of DeepSeek-R1-Zero on AIME 2024 rose from 15.6% to 71.0%.

Lesson 26 did the "reinforcement learning with a hand-written grader" part (GRPO-style), the closed-form solution
$\pi^*$, DPO on a table, and best-of-N. This topic adds what Lesson 26 doesn't have: **a reward learned from noisy
comparisons**, full PPO, and the price of optimizing against a reward that is only approximately right.

## 4 · From comparisons to scores: the Bradley–Terry model

Raters don't give scores, they only choose. The Bradley–Terry model connects the two: each response has a score $r$,
and
$$P(y_a \succ y_b) = \sigma\big(r(y_a) - r(y_b)\big), \qquad
  \mathcal{L}(r) = -\sum_{\text{pairs}} \log \sigma\big(r(y_{\text{winner}}) - r(y_{\text{loser}})\big).$$
Only the **difference** of two scores appears in the formula. The cell below fits the model for 6 responses to two
prompts (A: responses 0–2, B: responses 3–5), with 4,000 simulated comparisons, always between two responses to the
*same* prompt, as in real RLHF. We fit twice, from two different starting points, and compare with the "true quality"
$u$ that the rater uses.

In [ ]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def fit_bt(i, j, y, r0, steps=3000, lr=2.0):
    """Bradley–Terry by gradient descent; y = 1 when response i beats response j."""
    r = r0.astype(float).copy()
    for _ in range(steps):
        e = sigmoid(r[i] - r[j]) - y  # derivative of the loss with respect to (r_i − r_j), pair by pair
        r -= lr * (np.bincount(i, e, len(r)) - np.bincount(j, e, len(r))) / len(y)
    return r


def bt_nll(r, i, j, y):
    d = r[i] - r[j]
    return float(np.mean(np.logaddexp(0, -d) * y + np.logaddexp(0, d) * (1 - y)))


u = np.array([0.2, 1.0, 1.6, 0.4, 0.7, 2.0])
prompt_of = np.array([0, 0, 0, 1, 1, 1])
rng = np.random.default_rng(0)
grp = rng.integers(0, 2, 4000)
i = 3 * grp + rng.integers(0, 3, 4000)
j = 3 * grp + (i - 3 * grp + rng.integers(1, 3, 4000)) % 3  # another response to the same prompt
y = (rng.random(4000) < sigmoid(u[i] - u[j])).astype(float)
fit_a = fit_bt(i, j, y, np.zeros(6))
fit_b = fit_bt(i, j, y, np.array([5.0, 5, 5, -5, -5, -5]))
center = lambda r: r - np.array([r[prompt_of == p].mean() for p in prompt_of])
print("resp  prompt     u   fit from 0   fit from ±5   fit − prompt mean   u − prompt mean")
for k in range(6):
    print(
        f"{k:3}      {'AB'[prompt_of[k]]}    {num(u[k], 1):>4}   {num(fit_a[k]):>10}   {num(fit_b[k]):>11}"
        f"   {num(center(fit_a)[k]):>17}   {num(center(u)[k]):>15}"
    )
print("loss:", num(bt_nll(fit_a, i, j, y), 4), "and", num(bt_nll(fit_b, i, j, y), 4))
assert abs(bt_nll(fit_a, i, j, y) - bt_nll(fit_b, i, j, y)) < 1e-9
assert np.allclose(center(fit_a), center(fit_b)) and np.abs(center(fit_a) - center(u)).max() < 0.15

The two fits have the same loss to 4 digits, but the gap between the two prompts' scores differs by exactly 10 from
one fit to the other. After subtracting each prompt's mean, the two fits coincide and are close to $u$. The gradient
can never change that mean: the gradient summed over the responses to one prompt is always 0, because each pair adds
$e$ to one response and subtracts $e$ from the other.

What this means for RLHF:
- **Scores can only be compared between responses to the same prompt.** A neural-network reward model is no
  different: the offset between prompts is arbitrary. Adding a per-prompt constant to the reward doesn't change the
  optimal policy, but it makes the gradient noisier. That is why PPO needs a value model, while RLOO and GRPO
  subtract the mean over the group of responses to the same prompt.
- **The scale comes from the rater's noise.** The model learns $s\cdot u$, where $s$ says how sharply the rater tells
  responses apart. The noisier the rater, the closer together the scores. That is why we normalize every reward
  (section 8), so that the same β means the same thing for every reward model.

## 5 · The task: writing the eight-syllable line

### The lục bát rules as the rater's "taste"

In real RLHF, what people want can't be written down as a program. Here we *pretend* that the rater cares only about
the rules of lục bát (“six–eight”) verse, written as 7 checks on the response $y$ (the eight-syllable line, *câu
bát*) after the prompt $x$ (the six-syllable line, *câu lục*):
1. $y$ has exactly 8 syllables;
2. to 5. syllables 2, 4, 6, 8 are, in that order, **level, oblique, level, level** (a level tone, *bằng*: no tone
   mark, the ngang tone as in “a”, or the huyền mark, “à”; an oblique tone, *trắc*: sắc “á”, hỏi “ả”, ngã “ã”, nặng
   “ạ”);
6. syllable 6 of $y$ **rhymes** with syllable 6 of $x$ (a loose rhyme, *vần thông*: the same final sound, the coda,
   and the same vowel group, except that when the coda is *nh*, *ch* or *i* any vowel will do);
7. of syllables 6 and 8 of $y$, one has the huyền tone (falling) and the other the ngang tone (level, unmarked).

The **gold** score $g \in [0, 1]$ is the fraction of checks passed. It is the "true quality" that we use to
*measure*, never to train. The cell below writes the code that extracts a syllable's tone and rime, then tries it on
a few syllables (đường “road”, thuyền “boat”, lặng “still”, hoa “flower”, thương “love”, tình “love”, xanh “green”,
người “person”).

In [ ]:
TONE = {"̀": "huyền", "́": "sắc", "̃": "ngã", "̉": "hỏi", "̣": "nặng"}  # tone marks, NFD form
ONSET = re.compile(r"^(ngh|ng|nh|ch|gh|gi(?=[aeiouyăâêôơư])|kh|ph|th|tr|qu|[bcdđghklmnpqrstvx])")
CODAS = ("ng", "nh", "ch", "c", "m", "n", "p", "t", "i", "y", "o", "u")
NUCLEUS = {"y": "i", "yê": "iê", "ia": "iê", "ya": "iê", "ua": "uô", "ưa": "ươ"}  # spellings of the same vowel
GROUP = {"i": "F", "ê": "F", "e": "F", "iê": "F", "u": "B", "ô": "B", "o": "B", "uô": "B",
         "a": "A", "ă": "A", "â": "A", "ơ": "A", "ư": "A", "ươ": "A"}  # fmt: skip
FREE_CODA = {"nh", "ch", "i"}  # inh ~ anh ~ ênh, ai ~ ơi ~ ươi: any vowel rhymes


def tone(s):
    marks = [TONE[c] for c in unicodedata.normalize("NFD", s) if c in TONE]
    return marks[0] if marks else "ngang"


def rime(s):
    """(vowel, coda) of a syllable, tone mark removed: the part that makes the rhyme."""
    bare = unicodedata.normalize("NFC", "".join(c for c in unicodedata.normalize("NFD", s) if c not in TONE))
    r = ONSET.sub("", bare)
    if len(r) > 1 and r[0] in "ou" and r[1] in "aăâeêyơ":  # the glide o/u: hoa, thuyền, quân
        r = r[1:]
    nuc, coda = r, ""
    for c in CODAS:
        if r.endswith(c) and len(r) > len(c):
            nuc, coda = r[: -len(c)], c
            break
    return NUCLEUS.get(nuc, nuc), {"y": "i", "o": "u"}.get(coda, coda)


def rhymes(a, b):
    (na, ca), (nb, cb) = rime(a), rime(b)
    return ca == cb and (ca in FREE_CODA or GROUP.get(na, na) == GROUP.get(nb, nb))


for s in ("đường", "thuyền", "lặng", "hoa"):
    print(f"{s:7} tone {tone(s):6} rime {rime(s)}")
for a, b in (("thương", "đường"), ("tình", "xanh"), ("hoa", "người")):
    print(f"{a} – {b}: {'rhyme' if rhymes(a, b) else 'no rhyme'}")
assert rhymes("thương", "đường") and rhymes("tình", "xanh") and not rhymes("hoa", "người")

To grade thousands of lines at once, we precompute for each token: level or oblique tone, whether the tone is huyền,
the coda, and the vowel group. The seven checks are then just table lookups on tensors. The cell below grades *The
Tale of Kiều* itself: if the rules are written correctly, nearly every real eight-syllable line should pass all seven.

In [ ]:
SYL = torch.tensor([k > EOS for k in range(V)])
tones = [tone(s) if k > EOS else "" for k, s in enumerate(VOCAB)]
BANG = torch.tensor([t in ("ngang", "huyền") for t in tones]) & SYL
HUYEN = torch.tensor([t == "huyền" for t in tones])
rimes = [rime(s) if k > EOS else ("", "") for k, s in enumerate(VOCAB)]
codas, groups = sorted({c for _, c in rimes}), sorted({GROUP.get(n, n) for n, _ in rimes})
CODA = torch.tensor([codas.index(c) for _, c in rimes])
VGROUP = torch.tensor([groups.index(GROUP.get(n, n)) for n, _ in rimes])
FREE = torch.tensor([c in FREE_CODA for _, c in rimes])
CHECKS = [
    "8 syllables",
    "syl. 2 level",
    "syl. 4 oblique",
    "syl. 6 level",
    "syl. 8 level",
    "syl. 6 rhymes",
    "6–8 huyền/ngang",
]


def resp_len(seq):
    """Number of syllables before the first <eos>/<pad> of each response."""
    r = seq[:, PLEN:]
    return ((r != EOS) & (r != PAD)).int().cumprod(1).sum(1)


def checks(seq):
    """(B, 7) booleans: the 7 checks of each response (seq = prompt, <sep>, response, padding)."""
    r = torch.cat([seq[:, PLEN:], torch.full((len(seq), max(0, 8 - seq.shape[1] + PLEN)), PAD)], 1)
    n = resp_len(seq)
    y2, y4, y6, y8, x6 = r[:, 1], r[:, 3], r[:, 5], r[:, 7], seq[:, 5]
    return torch.stack([
        n == 8,
        (n >= 2) & BANG[y2],
        (n >= 4) & SYL[y4] & ~BANG[y4],
        (n >= 6) & BANG[y6],
        (n >= 8) & BANG[y8],
        (n >= 6) & (CODA[x6] == CODA[y6]) & (FREE[y6] | (VGROUP[x6] == VGROUP[y6])),
        (n >= 8) & BANG[y6] & BANG[y8] & (HUYEN[y6] != HUYEN[y8]),
    ], 1)  # fmt: skip


def gold(seq):
    return checks(seq).float().mean(1)


def padded(x):
    return torch.cat([x, torch.full((len(x), T_MAX - x.shape[1]), PAD)], 1)


real = checks(padded(X)).float().mean(0)
for name, v in zip(CHECKS, real.tolist()):
    print(f"  {name:16} {pct(v, 2)}")
assert real.min() >= 0.99 and np.allclose(real.tolist(), RUNS["setup"]["real"], atol=1e-4), (
    "differs from the saved results"
)

Three checks fall short of 100% because a few words in the poem are read the old way, or because the hand-written
rhyme rule doesn't catch every case. But rates close to 100% show that the 7 checks really are the rules that Nguyễn
Du followed.

### The SFT model

The policy is a very small GPT: 2 Transformer blocks, $d = 64$, 4 attention heads, and an output layer that shares its
matrix with the embedding layer (as GPT-2 does in Lesson 17). `sweep.py` trained it on the 1,300 training couplets,
picked the step with the lowest validation loss, then rounded the weights to float16. The cell below loads exactly
those weights. This is $\pi_{\text{ref}}$ for every experiment.

In [ ]:
D, HEADS = 64, 4


class Block(nn.Module):
    """Pre-LN Transformer block: causal self-attention, then a GELU MLP, each inside a skip connection."""

    def __init__(self):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(D), nn.LayerNorm(D)
        self.qkv, self.proj = nn.Linear(D, 3 * D), nn.Linear(D, D)
        self.fc, self.out = nn.Linear(D, 4 * D), nn.Linear(4 * D, D)

    def forward(self, h):
        B, T, _ = h.shape
        q, k, v = self.qkv(self.ln1(h)).view(B, T, 3, HEADS, D // HEADS).permute(2, 0, 3, 1, 4)
        a = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        h = h + self.proj(a.transpose(1, 2).reshape(B, T, D))
        return h + self.out(F.gelu(self.fc(self.ln2(h))))


class GPT(nn.Module):
    """A small GPT over syllables; the output layer is the embedding matrix. Never generates <pad> or <sep>."""

    def __init__(self):
        super().__init__()
        self.emb = nn.Embedding(V, D)
        self.pos = nn.Parameter(torch.zeros(T_MAX, D))
        self.blocks = nn.ModuleList(Block() for _ in range(2))
        self.ln = nn.LayerNorm(D)

    def hidden(self, x):
        h = self.emb(x) + self.pos[: x.shape[1]]
        for b in self.blocks:
            h = b(h)
        return self.ln(h)

    def forward(self, x):
        z = self.hidden(x) @ self.emb.weight.T
        return z.index_fill(-1, torch.tensor([PAD, SEP]), -1e9)  # exp(−1e9) is exactly 0


def unpack(m, d):
    """Load float16 weights (base64) into the model."""
    sd = m.state_dict()
    for k, v in d.items():
        a = np.frombuffer(base64.b64decode(v["f16"]), np.float16).astype(np.float32).reshape(v["shape"])
        sd[k] = torch.from_numpy(a.copy())
    m.load_state_dict(sd)
    return m


def frozen(m):
    m = copy.deepcopy(m)
    for p in m.parameters():
        p.requires_grad_(False)
    return m


sft = frozen(unpack(GPT(), W["sft"]))
n_params = sum(p.numel() for p in sft.parameters())
print(f"{num(n_params, 0)} parameters; SFT step chosen: {int(RUNS['setup']['sft']['step'])}")
assert n_params == RUNS["setup"]["params"] == 254016

We sample by the **inverse CDF** from random numbers drawn in advance from a seeded `torch.Generator`: each response
uses exactly 13 numbers, so the same seed always gives the same line, whether the line ends early or late. The
function `evaluate` has the policy write one eight-syllable line for each test prompt and measures: the gold score,
the pass rate of each check, the line's divergence $\mathrm{KL}(\pi\,\|\,\pi_{\text{ref}})$ (computed exactly over the
whole vocabulary at each step, then summed along the line), the length, and the fraction of distinct pairs of
adjacent syllables (distinct-2: low means the line repeats itself).

In [ ]:
def sample(model, prompts, gen, temp=1.0):
    """The response after each prompt (PLEN tokens), sampled by the inverse CDF; padded to T_MAX."""
    seq, done = prompts, torch.zeros(len(prompts), dtype=torch.bool)
    u = torch.rand(RESP, len(prompts), 1, generator=gen)
    with torch.no_grad():
        for t in range(RESP):
            if done.all():
                seq = torch.cat([seq, torch.full((len(seq), RESP - t), PAD)], 1)
                break
            p = torch.softmax(model(seq)[:, -1] / temp, -1)
            nxt = (p.cumsum(-1) < u[t]).sum(-1).clamp(max=p.shape[-1] - 1)
            nxt = torch.where(done, PAD, nxt)
            done = done | (nxt == EOS)
            seq = torch.cat([seq, nxt[:, None]], 1)
    return seq


def resp_mask(seq):
    """The targets seq[:, 1:] that belong to the response (syllables and <eos>, not padding)."""
    tgt = seq[:, 1:]
    m = torch.zeros_like(tgt, dtype=torch.bool)
    m[:, PLEN - 1 :] = True
    return m & (tgt != PAD)


def token_logp(model, seq):
    """log π(token | everything before it) for every target seq[:, 1:], and the whole log distribution."""
    lsm = torch.log_softmax(model(seq[:, :-1]), -1)
    return lsm.gather(-1, seq[:, 1:, None]).squeeze(-1), lsm


def seq_logp(model, seq):
    lp, _ = token_logp(model, seq)
    return (lp * resp_mask(seq)).sum(1)


def exact_kl(model, ref, seq):
    """KL(π‖π_ref) of each sampled line: exact over the vocabulary at each step, summed along the line."""
    with torch.no_grad():
        _, a = token_logp(model, seq)
        _, b = token_logp(ref, seq)
        per = (a.exp() * (a - b)).sum(-1)
    return (per * resp_mask(seq)).sum(1)


def decode(seq, k=None):
    out = []
    for row in seq[:k].tolist():
        resp = []
        for t in row[PLEN:]:
            if t in (EOS, PAD):
                break
            resp.append(VOCAB[t])
        out.append(" ".join(VOCAB[t] for t in row[: PLEN - 1]) + " / " + " ".join(resp))
    return out


def evaluate(pol, rewards=None, copies=1):
    """Measure on fresh samples of the policy for the 164 test prompts (always the same random stream, seed 2000)."""
    seq = sample(pol, X_test[:, :PLEN].repeat(copies, 1), torch.Generator().manual_seed(2000))
    c, n, r = checks(seq).float(), resp_len(seq), seq[:, PLEN:]
    pair = r[:, :-1] * V + r[:, 1:]
    bg = pair[torch.arange(RESP - 1)[None] < (n - 1)[:, None]]
    out = {"gold": c.mean().item(), "checks": c.mean(0).tolist(), "kl": exact_kl(pol, sft, seq).mean().item(),
           "len": n.float().mean().item(), "distinct2": len(set(bg.tolist())) / max(1, len(bg)), "top": []}  # fmt: skip
    for j in (1, 5, 7):  # the most frequent syllable at positions 2, 6, 8 of the response, and its share
        v, cnt = torch.unique(r[:, j], return_counts=True)
        out["top"].append([VOCAB[v[cnt.argmax()]], cnt.max().item() / len(r)])
    for k, f in (rewards or {}).items():
        out[k] = f(seq).mean().item()
    return out, seq


ev_sft, seq_sft = evaluate(sft)
print("\n".join(decode(seq_sft, 6)))
print(
    f"\ngold {num(ev_sft['gold'], 3)} · length {num(ev_sft['len'], 2)} syllables · "
    f"distinct-2 {num(ev_sft['distinct2'], 3)}"
)
for name, v in zip(CHECKS, ev_sft["checks"]):
    print(f"  {name:16} {pct(v)}")
saved = RUNS["live"]["check"]["sft"]
assert abs(ev_sft["gold"] - saved["gold"]) < 0.01, "differs from the saved results"

The lines of $\pi_{\text{ref}}$ have exactly the shape of an eight-syllable line: 95.7% have 8 syllables, and
syllables 2, 6 and 8 carry a level tone in 84.8–86.6% of the lines, because those are the things that repeat in every
eight-syllable line of the poem. But the lines make no sense: a model of 254,016 parameters learned from 1,300 lines
only picks up surface statistics. The two checks that have to *compare* two syllables rarely pass: syllable 6 rhymes
with the prompt in 12.8% of the lines, the huyền/ngang pair at syllables 6–8 appears in 36.0%. The gold score of 0.679
is where every experiment below starts.

### The simulated rater

For each comparison, we have $\pi_{\text{ref}}$ write two eight-syllable lines for the same prompt. The simulated
rater chooses by Bradley–Terry on the gold score, with a "sharpness" $s = 10$:
$$P(y_a \succ y_b) = \sigma\big(s\,(g_a - g_b)\big).$$
When two lines differ by one check ($\Delta g = 1/7$), the better one is chosen with probability $\sigma(10/7)$. Gao,
Schulman & Hilton (2022) also replaced people with such a "gold reward model" to measure Goodhart, but their labels
had no noise (they always chose the line with the higher gold score). We choose $s$ so that two independent raters
agree with each other about as often as InstructGPT's raters did. The cell below labels the 1,000 saved training pairs
again, with exactly the script's random stream.

In [ ]:
S = RUNS["setup"]["S"]


def unpack_pairs(d):
    a = torch.from_numpy(np.frombuffer(base64.b64decode(d["a"]), np.int16).astype(np.int64).reshape(-1, T_MAX))
    b = torch.from_numpy(np.frombuffer(base64.b64decode(d["b"]), np.int16).astype(np.int64).reshape(-1, T_MAX))
    return a, b, torch.tensor(d["y"], dtype=torch.float32)


def annotate(a, b, seed, split):
    """1 when the rater picks a: P(a ≻ b) = σ(S·(g_a − g_b))."""
    u = torch.rand(len(a), generator=torch.Generator().manual_seed(15_000 + 1000 * split + seed))
    return (u < torch.sigmoid(S * (gold(a) - gold(b)))).float()


pairs = {sp: unpack_pairs(W["pairs"][sp]) for sp in ("train", "val", "test")}
a_tr, b_tr, y_tr = pairs["train"]
assert torch.equal(annotate(a_tr, b_tr, 0, 0), y_tr), "the labels differ from the saved file"
ga, gb = gold(a_tr), gold(b_tr)
p = torch.sigmoid(S * (ga - gb))
print("one pair:", decode(a_tr[:1])[0], f"(gold {num(ga[0], 3)})")
print("         ", decode(b_tr[:1])[0], f"(gold {num(gb[0], 3)})")
print(f"P(first line chosen) = {num(p[0], 3)}; the rater chose the {'first' if y_tr[0] else 'second'} line")
print(f"\n1,000 pairs: tied gold score {pct((ga == gb).float().mean())}, "
      f"two independent raters agree {pct((p**2 + (1 - p) ** 2).mean())}")  # fmt: skip

### 🤔 Predict first
Suppose a *perfect* reward model: it knows the exact gold score of every line and always picks the line with the
higher $g$ (guessing at random on a tie). On the test pairs, what percentage of the rater's labels does it predict
correctly? 100%? 90%? Less?

In [ ]:
a_te, b_te, y_te = pairs["test"]
ga_t, gb_t = gold(a_te), gold(b_te)
p_t = torch.sigmoid(S * (ga_t - gb_t))
perfect = torch.where(ga_t == gb_t, 0.5, ((ga_t > gb_t).float() == y_te).float())
print(f"the perfect model predicts {pct(perfect.mean())} of the labels of the 1,000 test pairs correctly")
print(
    f"theoretical ceiling E[max(p, 1 − p)]: {pct(torch.maximum(p_t, 1 - p_t).mean())} "
    f"(on all 2,000 of the script's test pairs: {pct(RUNS['setup']['ceiling'])})"
)

Not 100%, but 79.4%. The rater's labels are inherently noisy: in the example pair above, the two lines differ by one
check, and the better one is chosen only with probability 0.807. When two lines tie on the gold score (21.6% of the
training pairs), the label is just a coin flip. In the long run, no model predicts the labels better than the ceiling
$\mathbb{E}[\max(p, 1-p)]$, 78.3% on these 1,000 pairs, not even a model that knows all the rules; 79.4% is slightly
above the ceiling only out of luck on a finite sample. Two independent raters also agree with each other only 73.2%
of the time, within InstructGPT's 72.6–77.3%. So a reward model's accuracy has to be read against this ceiling, not
against 100%.


## 6 · The reward model

The reward model is a copy of the SFT model with its output layer removed and replaced by a linear head that outputs
one number: the score of the whole line, read at the last token (`<eos>`). The embedding layer stays as in SFT
(frozen), just as InstructGPT initializes its reward model from the SFT model. The loss is section 4's Bradley–Terry
loss on the difference of the two lines' scores. The cell below trains it for 2 passes over 1,000 pairs, picks the
pass with the lowest loss on 500 validation pairs (by the rater's labels, never by gold), then measures it on 1,000
test pairs.

In [ ]:
class Scorer(nn.Module):
    """Reward model / value model: a copy of the SFT model's body (embedding layer frozen) and a scalar output head."""

    def __init__(self, base):
        super().__init__()
        self.body = copy.deepcopy(base)
        for p_ in self.body.parameters():
            p_.requires_grad_(True)
        self.body.emb.weight.requires_grad_(False)
        self.head = nn.Linear(D, 1)

    def values(self, seq):
        """One value at each position, aligned with the targets seq[:, 1:] (used for PPO's value model)."""
        return self.head(self.body.hidden(seq[:, :-1])).squeeze(-1)

    def forward(self, seq):
        """Score of the whole line: the output at the last token that isn't padding (<eos>)."""
        h = self.body.hidden(seq)
        last = (seq != PAD).sum(1) - 1
        return self.head(h[torch.arange(len(seq)), last]).squeeze(-1)


def bt_loss(rm, a, b, y):
    return F.binary_cross_entropy_with_logits(rm(a) - rm(b), y)


def train_rm(a, b, y, va, seed, epochs, lr=1e-3, batch=64):
    torch.manual_seed(seed)
    rm = Scorer(sft)
    opt = torch.optim.Adam([p for p in rm.parameters() if p.requires_grad], lr)
    g = torch.Generator().manual_seed(20_000 + seed)
    best, curve = (float("inf"), None, 0), []
    for ep in range(1, epochs + 1):
        for j in torch.randperm(len(y), generator=g).split(batch):
            loss = bt_loss(rm, a[j], b[j], y[j])
            opt.zero_grad()
            loss.backward()
            opt.step()
        with torch.no_grad():
            vl = bt_loss(rm, *va).item()
        curve.append((ep, vl))
        if vl < best[0]:
            best = (vl, copy.deepcopy(rm.state_dict()), ep)
    rm.load_state_dict(best[1])
    with torch.no_grad():  # round to float16, as the script does
        for p in rm.parameters():
            p.copy_(p.half().float())
    return frozen(rm), curve


def rm_accuracy(rm, a, b, y):
    with torch.no_grad():
        d = rm(a) - rm(b)
    g_a, g_b = gold(a), gold(b)
    return ((d > 0).float() == y).float().mean().item(), ((d > 0) == (g_a > g_b))[g_a != g_b].float().mean().item()


rm_live, curve = train_rm(a_tr, b_tr, y_tr, pairs["val"], 0, RUNS["live"]["cfg"]["rmEpochs"])
acc, gacc = rm_accuracy(rm_live, a_te, b_te, y_te)
print("validation loss by pass:", ", ".join(num(v, 4) for _, v in curve))
print(f"matches the rater's labels {pct(acc)} · matches the gold order (ties excluded) {pct(gacc)}")
assert abs(acc - RUNS["live"]["rm"]["acc"]) < 0.01, "differs from the saved results"

After 2 passes over 1,000 pairs, the model predicts 61.2% of the labels correctly (ceiling 78.3%) and gets the gold
order right in 66.8% of the pairs without a tie: clearly better than guessing, but still far from the ceiling.

The script trained the reward model on 250 to 16,000 pairs, 3 seeds for each size, for at most 8 passes (the pass
chosen on the validation pairs). The figure below plots the accuracy on 2,000 test pairs against the number of pairs:
matching the rater's labels (circles, blue), matching the gold order (squares, orange), against the ceiling (dashed
line). The cell also prints the correlation coefficient between the model's score and the gold score on the lines of
$\pi_{\text{ref}}$.

In [ ]:
NS = sorted({x["n"] for x in RUNS["rm"] if x["kind"] == "plain"})


def rm_rows(kind, n):
    return [x for x in RUNS["rm"] if x["kind"] == kind and x["n"] == n]


fig, ax = plt.subplots(figsize=(7, 3.6))
for key, label, color, marker in (("acc", "matches the rater's labels", BLUE, "o"),
                                  ("goldAcc", "matches the gold order", ORANGE, "s")):  # fmt: skip
    v = np.array([[x[key] for x in rm_rows("plain", n)] for n in NS])
    ax.plot(NS, v.mean(1), color=color, marker=marker, label=label)
    ax.fill_between(NS, v.min(1), v.max(1), color=color, alpha=0.2)
ax.axhline(RUNS["setup"]["ceiling"], color=GREY, ls="--", label="ceiling for matching labels (noisy labels)")
ax.set(xscale="log", xlabel="comparison pairs used for training", ylabel="accuracy on test pairs",
       title="Reward model: the more comparisons, the more accurate")  # fmt: skip
ax.legend(frameon=False)
ax.minorticks_off()
fmt_axes(fig)
ax.set_xticks(NS, [num(n, 0) for n in NS])  # after fmt_axes, to keep the ticks at the four measured sizes
plt.show()
for n in NS:
    rows = rm_rows("plain", n)
    print(
        f"{num(n, 0):>6} pairs: labels correct {pct(min(x['acc'] for x in rows))}–{pct(max(x['acc'] for x in rows))}, "
        f"correlation with gold {num(min(x['corr'] for x in rows), 2)}–{num(max(x['corr'] for x in rows), 2)}"
    )

Accuracy rises steadily with the number of pairs, from 53.5–61.9% (250 pairs) to 69.3–71.2% (16,000 pairs), and stays
below the ceiling of 78.0%. InstructGPT reports 72.4% on the labels of the very raters who provided the training data,
69.6% with new raters. The orange curve goes above the ceiling because it measures something else: the gold order,
not the noisy labels. The reward model is better at guessing *which line is truly better* than at guessing *what the
rater chose*. The correlation with gold also rises, from 0.35–0.49 to 0.74–0.76.

From here on, every experiment optimizes against the seed-0 reward model trained on **4,000 pairs**, which the
weights file includes. What does it see in a line of verse? The cell below has $\pi_{\text{ref}}$ write 16 lines for
each of 64 test prompts, scores all 1,024 lines, then fits a linear regression of the reward model's score on the 7
checks: each coefficient is the number of points the model "adds" when a check passes.

In [ ]:
rm = frozen(unpack(Scorer(sft), W["rm"]))  # the file has no embedding layer: it is exactly the SFT model's
cfg = RUNS["live"]["cfg"]
P_bon = X_test[: cfg["bonPrompts"], :PLEN]
seq_bon = sample(sft, P_bon.repeat_interleave(cfg["bonN"], 0), torch.Generator().manual_seed(cfg["bonSeed"]))
with torch.no_grad():
    raw_score = rm(seq_bon)
C = checks(seq_bon).float()
coef = torch.linalg.lstsq(torch.cat([C, torch.ones(len(C), 1)], 1), raw_score[:, None]).solution.squeeze()
for name, w, share in zip(CHECKS, coef.tolist(), C.mean(0).tolist()):
    print(f"  {name:16} {num(w, 2):>6}   (passed by {pct(share)} of SFT lines)")
print("correlation with gold:", num(np.corrcoef(raw_score.numpy(), gold(seq_bon).numpy())[0, 1], 3))

The reward model adds 0.56 to 1.10 points for each length and tone rule, but almost 0 for the remaining two rules: the
rhyme (0.03) and the huyền/ngang pair at syllables 6–8 (−0.02). These two rules have to *compare* two syllables
(syllable 6 with syllable 6 of the prompt, syllable 6 with syllable 8), and SFT lines rarely pass them, so 4,000 noisy
comparisons carry little information about them. The correlation with gold is still fairly high (0.674), because the
five easy rules account for most of the differences between SFT lines. But this is a blind spot, and optimization
will find it.

## 7 · Best-of-N and Goodhart's law

The simplest way to "optimize against" the reward model is to train nothing at all: write $N$ lines and keep the one
with the highest score. Best-of-N drifts from $\pi_{\text{ref}}$ by at most $\log N - (N-1)/N$ nats (computed in
Lesson 26), so we have a common axis for every way of optimizing: **the KL divergence**. Gao, Schulman & Hilton (2022)
had InstructGPT's 6-billion-parameter reward model play the role of "gold", trained smaller reward models from its
labels, then optimized a 1.2-billion-parameter policy against them. Plotted against $d = \sqrt{\mathrm{KL}}$, the gold
score rises and then falls, while the score of the learned reward model (called the *proxy*) keeps rising. They fit
two functional forms to the gain in gold: $d\,(a - b\,d)$ for best-of-N and $d\,(a - b \log d)$ for reinforcement
learning (they call the coefficients α, β; we rename them because β here is the KL penalty coefficient).

### On a table, computed exactly: the tail of the error

Why does gold *fall*? The proxy equals gold plus an error, so optimizing the proxy picks responses with a high gold
*or* a large error. Which of the two wins depends on the shape of the error (Kwa, Thomas & Garriga-Alonso, 2024). The
cell below takes one prompt with 5,000 responses, a uniform $\pi_{\text{ref}}$, gold $g \sim \mathcal{N}(0, 1)$, and
two reward models $r = g + \varepsilon$. Their errors have the same typical size but different **tails**: a normal
$\varepsilon$ (light tail), and a Student-t $\varepsilon$ with ν = 1.5 (heavy tail: now and then wildly off). With a
table, everything can be computed exactly: the optimal KL-penalized policy $\pi_\beta \propto \pi_{\text{ref}}\,
e^{r/\beta}$ (Lesson 26), the probability $(k/K)^N - ((k-1)/K)^N$ that best-of-N picks the response ranked $k$ (in
increasing order of $r$), and the KL of both.

In [ ]:
K = 5000
rng = np.random.default_rng(0)
g_tab = rng.standard_normal(K)
ERRORS = {"Light-tailed": rng.standard_normal(K), "Heavy-tailed": 0.6 * rng.standard_t(1.5, K)}


def tilt(r, beta):
    z = r / beta  # uniform π_ref: log π_ref is a constant
    q = np.exp(z - z.max())
    return q / q.sum()


def kl_unif(q):
    m = q > 0  # convention: 0·log 0 = 0
    return float(np.sum(q[m] * np.log(q[m] * K)))


def best_of(r, N):
    q, k = np.zeros(K), np.arange(1, K + 1)
    q[np.argsort(r)] = (k / K) ** N - ((k - 1) / K) ** N
    return q


BETAS_TAB, NS_TAB = np.geomspace(100, 0.02, 40), 2 ** np.arange(13)
table = {}
for name, eps in ERRORS.items():
    r_tab = g_tab + eps
    table[name] = {}
    for how, qs in (("rl", [tilt(r_tab, b) for b in BETAS_TAB]), ("bon", [best_of(r_tab, N) for N in NS_TAB])):
        table[name][how] = [np.array([kl_unif(q) for q in qs]), np.array(qs) @ g_tab, np.array(qs) @ r_tab]
    (_, g_r, px_r), (_, g_b, px_b) = table[name]["rl"], table[name]["bon"]
    print(f"{name}: median |ε| {num(np.median(abs(eps)), 2)}, largest {num(abs(eps).max(), 1)}")
    print(f"  best-of-N: highest gold {num(g_b.max(), 2)} at N = {num(NS_TAB[g_b.argmax()], 0)}; at N = 4,096 gold "
          f"{num(g_b[-1], 2)}, proxy {num(px_b[-1], 1)}")  # fmt: skip
    print(f"  π_β: highest gold {num(g_r.max(), 2)} at β = {num(BETAS_TAB[g_r.argmax()], 2)}; at β = 0.02 gold "
          f"{num(g_r[-1], 2)}, proxy {num(px_r[-1], 1)}")  # fmt: skip
assert abs(kl_unif(best_of(g_tab, 16)) - (np.log(16) - 15 / 16)) < 0.01  # the KL of best-of-N is close to the bound

The figure below plots the gold score against $\sqrt{\mathrm{KL}}$: the blue circles are $\pi_\beta$ (β decreasing
from left to right), the orange squares best-of-N ($N = 1$ to 4,096), the dotted line the gold of $\pi_{\text{ref}}$.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
for ax, name in zip(axes, ERRORS):
    for how, color, marker, label in (("rl", BLUE, "o", "π_β"), ("bon", ORANGE, "s", "best-of-N")):
        kl_, g_, _ = table[name][how]
        ax.plot(np.sqrt(np.maximum(kl_, 0)), g_, color=color, marker=marker, ms=4, label=label)
    ax.axhline(g_tab.mean(), color=GREY, ls=":", lw=1)
    ax.set(xlabel="√KL(π‖π_ref)", title=f"{name} error")
axes[0].set_ylabel("gold score")
axes[0].legend(frameon=False)
fig.suptitle("The same gold, two reward models: only the tail of the error differs", y=1.02)
fmt_axes(fig)
plt.show()

**Light tail:** gold rises all the way (to 2.73 with $\pi_\beta$ at β = 0.02, and to 2.59 with best-of-N at
N = 4,096), though more slowly than the proxy (4.8 and 4.6). A response is picked because its gold is high *and*
because its error is positive, so the proxy always exaggerates its quality (the winner's curse), but optimizing
further still pays off.

**Heavy tail:** best-of-N raises gold up to N = 16 (0.89), then gold falls, down to −0.91 at N = 4,096, below even
$\pi_{\text{ref}}$, while the proxy climbs to 118.5. $\pi_\beta$ is even worse: the response with an error of 156.3
soaks up almost all the probability already at very large β, so the highest gold is only 0.01 (at β = 51.94). Kwa et
al. prove that this holds in general: when the error is heavy-tailed, there are policies that reach an arbitrarily
high proxy with no more gold than the original policy, at an arbitrarily small KL. The KL penalty doesn't prevent
that; they call it *catastrophic Goodhart*.

The error of a real reward model is not random noise. Section 6 showed that it ignores two rules entirely, so every
line that passes the five easy rules and breaks the other two is scored above its true value, systematically.
Optimization will home in on exactly those lines.

### Best-of-N on lục bát verse

The cell below reuses the 1,024 SFT lines of section 6 (16 lines for each of 64 prompts), scores them with the
4,000-pair reward model, and computes the expected gold score of best-of-N for $N = 1$ to 16 with the exact formula
for a finite pool of samples (drawing $N$ lines without replacement from the 16, as Gao et al. do): the line ranked
$i$ (in increasing order) is the best of the $N$ lines with probability $\binom{i-1}{N-1} / \binom{16}{N}$. The proxy
scores here are already normalized (section 8).

In [ ]:
def bon_curve(score, Ns, **metrics):
    """E[metric of the highest-scoring line among N], N lines drawn without replacement from each prompt's n lines."""
    n = score.shape[1]
    order = np.argsort(score, 1, kind="stable")
    cols = {"score": score, **metrics}
    sorted_ = {k: np.take_along_axis(v, order, 1) for k, v in cols.items()}
    out = []
    for N in Ns:
        w = np.array([math.comb(i - 1, N - 1) for i in range(1, n + 1)], float) / math.comb(n, N)
        row = {"N": int(N), "klBound": float(np.log(N) - (N - 1) / N)}
        row.update({k: float((v * w).sum(1).mean()) for k, v in sorted_.items()})
        out.append(row)
    return out


CHECK = RUNS["live"]["check"]
bon_live = bon_curve(((raw_score - CHECK["rmMu"]) / CHECK["rmSd"]).view(len(P_bon), -1).numpy(), [1, 2, 4, 8, 16],
                     gold=gold(seq_bon).view(len(P_bon), -1).numpy())  # fmt: skip
print(" N   KL ≤    proxy   gold")
for row in bon_live:
    print(f"{row['N']:2}   {num(row['klBound'], 2)}   {num(row['score'], 2):>6}   {num(row['gold'], 3)}")
assert abs(bon_live[-1]["gold"] - RUNS["live"]["bon"][-1]["gold"]) < 0.01, "differs from the saved results"

Picking 1 line out of 16 raises gold from 0.682 to 0.797, at a KL of at most 1.84 nats, without changing a single
weight.

The script does the same computation with 256 lines for each of the 164 test prompts, for each reward model (250 to
16,000 pairs) and for gold itself, 3 seeds. The figure below plots gold against $\sqrt{\mathrm{KL}}$ (using the upper
bound $\log N - (N-1)/N$, very close to the true KL when few lines repeat): each curve is one reward model, the band
is the range across seeds.

In [ ]:
def seeds_of(rows, key):
    return np.array([[e[key] for e in row] for row in rows])


fig, ax = plt.subplots(figsize=(7.5, 4))
styles = {"rm250": (GREY, "v", "250 pairs"), "rm1000": (GREEN, "^", "1,000 pairs"),
          "rm4000": (BLUE, "o", "4,000 pairs"), "rm16000": (ORANGE, "s", "16,000 pairs"),
          "gold": ("black", "*", "gold itself")}  # fmt: skip
for name, (color, marker, label) in styles.items():
    rows = [b["curves"][name] for b in RUNS["bon"]]
    d = np.sqrt([e["klBound"] for e in rows[0]])
    v = seeds_of(rows, "gold")
    ax.plot(d, v.mean(0), color=color, marker=marker, ms=5, label=label)
    ax.fill_between(d, v.min(0), v.max(0), color=color, alpha=0.15)
ax.set(
    xlabel="√KL (best-of-N upper bound)",
    ylabel="gold score",
    title="Best-of-N on 164 test prompts, N = 1 to 256",
)
ax.legend(frameon=False, title="picked by", fontsize=8)
fmt_axes(fig)
plt.show()
for name in styles:
    v = seeds_of([b["curves"][name] for b in RUNS["bon"]], "gold")
    k = v.mean(0).argmax()
    print(
        f"{styles[name][2]:>12}: N = 1: {num(v[:, 0].mean(), 3)} · highest (seed mean) at N = "
        f"{int(RUNS['bon'][0]['curves'][name][k]['N'])}: {num(v[:, k].min(), 3)}–{num(v[:, k].max(), 3)}"
        f" · N = 256: {num(v[:, -1].min(), 3)}–{num(v[:, -1].max(), 3)}"
    )

With the 4,000-pair reward model, gold reaches 0.788–0.796 at N = 64, then stays flat up to N = 256 (0.783–0.798). At
this light level of optimization, with every reward model, gold has not yet fallen by more than the variation between
seeds. The more pairs a model learned from, the higher its curve: with 16,000 pairs, gold rises to 0.789–0.823
at N = 256. Picking by gold itself takes it to 0.996. The gap between the black curve and the colored ones is the
price of having only an approximate reward model.

## 8 · PPO piece by piece

Best-of-N costs $N$ generations for each response. Reinforcement learning instead puts the optimization into the
weights themselves. InstructGPT used PPO (Schulman et al., 2017) with the components below; we assemble them one at a
time.

**Normalized reward.** By section 4, the reward model's scale is arbitrary. We subtract its mean and divide by its
standard deviation over SFT lines written for the validation prompts, so that "1" means one standard deviation better
than the average SFT line, for every reward model (and for gold).

In [ ]:
class Reward:
    """A reward function on lines, normalized on SFT lines for the validation prompts (4 lines per prompt)."""

    def __init__(self, fn, seed=0):
        self.fn = fn
        with torch.no_grad():
            s = fn(sample(sft, X_val[:, :PLEN].repeat(4, 1), torch.Generator().manual_seed(30_000 + seed)))
        self.mu, self.sd = s.mean().item(), s.std().item()

    def __call__(self, seq):
        with torch.no_grad():
            return (self.fn(seq) - self.mu) / self.sd


R = Reward(rm)
print(f"raw reward-model score on SFT lines: mean {num(R.mu, 3)}, standard deviation {num(R.sd, 3)}")
assert abs(R.mu - CHECK["rmMu"]) < 1e-3 and abs(R.sd - CHECK["rmSd"]) < 1e-3, "differs from the saved results"

**Per-token reward.** Each response is a sequence of actions (one syllable each). The KL penalty is paid at *every*
step, while the reward model's score arrives only at the last token:
$$r_t = -\beta\,\big(\log\pi(y_t \mid \cdot) - \log\pi_{\text{ref}}(y_t \mid \cdot)\big) + [\,t = \text{last}\,]\cdot r_\phi(x, y).$$
Summed along the line, the penalty is $\beta$ times a one-sample estimate of $\mathrm{KL}(\pi\,\|\,\pi_{\text{ref}})$.

**Value model and GAE.** A second model, $V(s_t)$, predicts the total reward still to come from step $t$; InstructGPT
initializes it from the reward model, and so do we. The advantage of each token by GAE (Schulman et al., 2016), with
$\gamma = 1$ and $\lambda = 0.95$:
$$\delta_t = r_t + V(s_{t+1}) - V(s_t), \qquad \hat A_t = \delta_t + \lambda\,\hat A_{t+1}.$$
$\lambda = 1$ gives the Monte Carlo advantage (little bias, a lot of noise); $\lambda = 0$ trusts only $V$ (little
noise, biased by the errors of $V$). The cell below computes all of it for one line of a trial policy, slightly
different from $\pi_{\text{ref}}$ so that the KL penalty isn't 0, and prints each token.

In [ ]:
def critic_from(scorer, reward):
    """The value model starts as the reward model, rescaled to the normalized reward."""
    c = copy.deepcopy(scorer)
    with torch.no_grad():
        c.head.weight /= reward.sd
        c.head.bias.sub_(reward.mu).div_(reward.sd)
    return c


def rollout(pol, crit, reward, beta, prompts, g):
    seq = sample(pol, prompts, g)
    m = resp_mask(seq).float()
    with torch.no_grad():
        lp_old, _ = token_logp(pol, seq)
        lp_ref, _ = token_logp(sft, seq)
        v_old = crit.values(seq) * m
        rew = -beta * (lp_old - lp_ref) * m
        last = (m * torch.arange(m.shape[1])).argmax(1)
        score = reward(seq)
        rew[torch.arange(len(seq)), last] += score
    return seq, m, lp_old, lp_ref, v_old, rew, score


def gae(rew, v, m, lam=0.95):
    """GAE advantages, γ = 1; nothing comes after the last token (no bootstrapping with V)."""
    adv, nxt_v, nxt_a = torch.zeros_like(rew), torch.zeros(len(rew)), torch.zeros(len(rew))
    for t in reversed(range(m.shape[1])):
        delta = rew[:, t] + nxt_v - v[:, t]
        nxt_a = (delta + lam * nxt_a) * m[:, t]
        adv[:, t], nxt_v = nxt_a, v[:, t]
    return adv


crit0 = critic_from(rm, R)
demo_pol = copy.deepcopy(sft)  # a policy slightly different from π_ref, so that the KL penalty isn't 0
with torch.no_grad():
    demo_pol.blocks[1].out.bias.add_(0.05 * torch.randn(D, generator=torch.Generator().manual_seed(1)))
seq, m, lp_old, lp_ref, v_old, rew, score = rollout(demo_pol, crit0, R, 0.1, X_test[:1, :PLEN],
                                                    torch.Generator().manual_seed(5))  # fmt: skip
adv = gae(rew, v_old, m)
print(decode(seq)[0], f"   (normalized score {num(score[0], 3)})")
print("token      log π − log π_ref   r_t      V(s_t)   Â_t")
for t in torch.nonzero(m[0]).squeeze(1).tolist():
    tok = VOCAB[seq[0, t + 1]]
    print(
        f"{tok:9} {num(lp_old[0, t] - lp_ref[0, t], 3):>12}   {num(rew[0, t], 3):>7}   {num(v_old[0, t], 3):>6}  "
        f"{num(adv[0, t], 3):>6}"
    )

Reading the table. The trial policy differs only slightly from $\pi_{\text{ref}}$, so the KL penalty in $r_t$ is
almost 0, and nearly all the reward arrives at `<eos>` (0.228). The $V(s_t)$ column is odd: from −6.250 to −1.298,
while the whole line only gets 0.228. The value model starts from the reward model, and the reward model has only
learned to score at the `<eos>` token; at the positions before it, its output doesn't mean anything yet. At first, the
advantage $\hat A_t$ therefore mostly measures the error of $V$. The value model learns over the PPO iterations, and
normalizing the advantages over the whole iteration removes part of the common offset.

**The clipped ratio.** Each iteration, PPO takes 128 lines from the current policy, then uses them for 4 update passes
(each pass 4 mini-batches of 32 lines). After the first pass, the policy already differs from the policy that
sampled the lines, so the objective uses the ratio $\rho_t = \pi(y_t)/\pi_{\text{old}}(y_t)$ and clips it to
$[1 - \varepsilon, 1 + \varepsilon]$, $\varepsilon = 0.2$:
$$\mathcal{L}^{\text{clip}} = -\,\mathbb{E}_t\Big[\min\big(\rho_t \hat A_t,\; \mathrm{clip}(\rho_t, 1-\varepsilon, 1+\varepsilon)\,\hat A_t\big)\Big].$$
When $\rho_t = 1$, its gradient is exactly the REINFORCE gradient of Lesson 26. The value model learns by squared
error against $\hat A_t + V_{\text{old}}(s_t)$. The advantages are normalized over all the tokens of the iteration. We
track two metrics: the fraction of tokens clipped, and the approximate KL between the old and the new policy; both
tell whether the learning rate η is too large.

The cell below is the whole loop. It runs 8 iterations (64 lines per iteration instead of 128, for speed), β = 0.01,
on the 4,000-pair reward model, and prints statistics of the very lines sampled at each iteration.

In [ ]:
def ppo(reward, critic0, beta, seed, iters, lr=3e-4, lr_v=1e-3, prompts=128, epochs=4, minibatch=32, clip=0.2):
    torch.manual_seed(seed)
    pol, crit = copy.deepcopy(sft), copy.deepcopy(critic0)
    for p_ in (*pol.parameters(), *crit.parameters()):
        p_.requires_grad_(True)
    crit.body.emb.weight.requires_grad_(False)
    opt_p = torch.optim.Adam(pol.parameters(), lr)
    opt_v = torch.optim.Adam([p_ for p_ in crit.parameters() if p_.requires_grad], lr_v)
    g = torch.Generator().manual_seed(40_000 + seed)
    trace = []
    for it in range(1, iters + 1):
        P = X_train[torch.randint(0, len(X_train), (prompts,), generator=g), :PLEN]
        seq, m, lp_old, lp_ref, v_old, rew, score = rollout(pol, crit, reward, beta, P, g)
        adv = gae(rew, v_old, m)
        ret = adv + v_old
        a = adv[m.bool()]
        adv = (adv - a.mean()) / (a.std() + 1e-8) * m
        stats = []
        for _ in range(epochs):
            for j in torch.randperm(len(seq), generator=g).split(minibatch):
                lp, _ = token_logp(pol, seq[j])
                ratio = torch.exp(lp - lp_old[j])
                mj = m[j]
                obj = torch.min(ratio * adv[j], torch.clamp(ratio, 1 - clip, 1 + clip) * adv[j])
                loss_p = -(obj * mj).sum() / mj.sum()
                opt_p.zero_grad()
                loss_p.backward()
                nn.utils.clip_grad_norm_(pol.parameters(), 1.0)
                opt_p.step()
                loss_v = (((crit.values(seq[j]) - ret[j]) * mj) ** 2).sum() / mj.sum()
                opt_v.zero_grad()
                loss_v.backward()
                nn.utils.clip_grad_norm_(crit.parameters(), 1.0)
                opt_v.step()
                with torch.no_grad():
                    lr_ = (lp - lp_old[j]).detach()
                    stats.append([(((ratio - 1).abs() > clip).float() * mj).sum().item() / mj.sum().item(),
                                  (((lr_.exp() - 1) - lr_) * mj).sum().item() / mj.sum().item()])  # fmt: skip
        clipfrac, approxkl = np.mean(stats, 0)
        trace.append({"it": it, "score": score.mean().item(), "gold": gold(seq).mean().item(),
                      "kl": ((lp_old - lp_ref) * m).sum(1).mean().item(), "clipfrac": clipfrac, "approxkl": approxkl})  # fmt: skip
    return pol, trace


pol_live, trace = ppo(R, crit0, cfg["beta"], 0, cfg["ppoIters"], prompts=cfg["ppoPrompts"])
print("iter   proxy   gold      line KL   clipped  approx. KL (old→new)")
for e in trace:
    print(
        f"{e['it']:4}   {num(e['score'], 2):>5}   {num(e['gold'], 3)}   {num(e['kl'], 2):>9}   {pct(e['clipfrac']):>6}"
        f"   {num(e['approxkl'], 4)}"
    )
assert all(abs(e["gold"] - s["gold"]) < 0.03 for e, s in zip(trace, RUNS["live"]["ppo"])), (
    "differs from the saved results"
)

In 8 iterations, the proxy score of the lines just sampled rises from −0.04 (iteration 1) to 0.04–0.30, their gold
from 0.665 to 0.692–0.721, and the line's KL from 0 to 0.65 nats. Each iteration has only 64 lines, so these numbers
are noisy. The fraction of clipped tokens is 9.1% and 6.7% in the first two iterations, then 1.3–3.0%, and the
approximate KL between the old and the new policy stays below 0.01: each iteration changes the policy only a little,
exactly what PPO wants. The script chose the learning rate before running, by these two metrics, not by gold.

### Where to put the KL?

There are two ways to bring $\beta\,\mathrm{KL}$ into the code: add it to the **reward** (as above, and as InstructGPT
does), or add it to the **loss** and let autograd take the derivative (as GRPO and many libraries do). The second way
needs an estimate of the KL from samples. Two common estimators, with $\rho = \pi_{\text{ref}}(y)/\pi(y)$ and
$y \sim \pi$: $k_1 = -\log\rho$ (unbiased, can be negative) and $k_3 = \rho - 1 - \log\rho$ (unbiased, always
$\geq 0$). Both *estimate* the value of the KL correctly. The question is their *derivatives* (Tang & Munos, 2025).

Let's try it on a bandit problem: 100 actions, a random reward $r$ in $[0, 1]$, β = 0.1, and a policy that is the
softmax of 100 logits starting at $\pi_{\text{ref}}$; each step takes 256 samples and updates with Adam. The right
target is $\pi^* \propto \pi_{\text{ref}}\,e^{r/\beta}$.

### 🤔 Predict first
The loss is REINFORCE plus $\beta \cdot \text{mean}(k_1)$ over the samples. After 2,000 steps, is the policy close to
$\pi^*$? If not, where does it head?

In [ ]:
torch.manual_seed(0)
logit_ref, r_band = torch.randn(100), torch.rand(100)
lp_ref_b = torch.log_softmax(logit_ref, 0)
beta_b = 0.1
star = torch.softmax(lp_ref_b + r_band / beta_b, 0)


def pi_rev(beta):
    """π_rev = β·π_ref/(λ − r), with λ > max r chosen by bisection so that the sum is 1."""
    lo, hi = r_band.max().item() + 1e-12, r_band.max().item() + 10
    for _ in range(200):
        lam_ = (lo + hi) / 2
        lo, hi = (lam_, hi) if (beta * lp_ref_b.exp() / (lam_ - r_band)).sum() > 1 else (lo, lam_)
    rev_ = beta * lp_ref_b.exp() / (lam_ - r_band)
    return rev_ / rev_.sum()


rev = pi_rev(beta_b)


def bandit(kind, steps=2000, batch=256, lr=0.05):
    g = torch.Generator().manual_seed(1)
    th = lp_ref_b.clone().requires_grad_(True)
    opt = torch.optim.Adam([th], lr)
    for _ in range(steps):
        lp = torch.log_softmax(th, 0)
        a = (lp.exp().cumsum(0)[None] < torch.rand(batch, 1, generator=g)).sum(1).clamp(max=99)
        log_ratio = lp[a] - lp_ref_b[a]  # log π(a) − log π_ref(a) = −log ρ
        if kind == "in the reward":
            Rk = (r_band[a] - beta_b * log_ratio).detach()
            loss = -((Rk - Rk.mean()) * lp[a]).mean()
        else:
            loss = -((r_band[a] - r_band[a].mean()) * lp[a]).mean()
            rho = torch.exp(-log_ratio)
            loss = loss + beta_b * (log_ratio if kind == "k1 in the loss" else rho - 1 - torch.log(rho)).mean()
        opt.zero_grad()
        loss.backward()
        opt.step()
    return torch.softmax(th.detach(), 0)


tv = lambda p_, q_: 0.5 * (p_ - q_).abs().sum().item()
band = {k: bandit(k) for k in ("in the reward", "k1 in the loss", "k3 in the loss")}
print(f"π_ref is {num(tv(lp_ref_b.exp(), star), 3)} away from π* (total variation distance, TV)")
for k, q in band.items():
    print(
        f"KL {k:14}: TV to π* {num(tv(q, star), 3)} · to π_rev {num(tv(q, rev), 3)} · "
        f"largest probability {pct(q.max())}"
    )

**KL in the reward:** the distance to π* is 0.000. At π*, the reward minus the KL penalty,
$r - \beta\log(\pi^*/\pi_{\text{ref}})$, is the same for every action, so the advantage is 0 on every sample and the
gradient vanishes entirely.

**$k_1$ in the loss:** the policy piles 99.8% of the probability onto a single action, as if there were no KL
penalty. The gradient of $k_1 = \log\pi(a) - \log\pi_{\text{ref}}(a)$ with respect to the parameters is
$\nabla\log\pi(a)$, and its expectation for $a \sim \pi$ is 0 (because $\sum_a \nabla\pi(a) = 0$). The estimate is
right, but its derivative is not the derivative of the KL.

**$k_3$ in the loss:** the policy converges, but to $\pi_{\text{rev}}$ (distance 0.024), not to π* (0.520). Taking the
expectation of the derivative $(1-\rho)\nabla\log\pi(a)$ gives $\nabla\,\mathrm{KL}(\pi_{\text{ref}}\,\|\,\pi)$: the KL
in the reverse direction. Maximizing $\mathbb{E}_\pi[r] - \beta\,\mathrm{KL}(\pi_{\text{ref}}\|\pi)$ gives
$\pi_{\text{rev}} = \beta\,\pi_{\text{ref}}/(\lambda - r)$, with λ chosen so that the sum is 1; the cell above finds λ
by bisection. DeepSeekMath's GRPO (Shao et al., 2024) adds exactly $k_3$ to the loss, so it actually penalizes the KL
in this direction. Lesson 26, on the other hand, puts the KL penalty in the reward, the first way. (Topic 03 also
meets the direction $\mathrm{KL}(\pi_0\,\|\,\pi)$: RL's Razor measures forgetting with it.)

## 9 · Long PPO runs: over-optimization and gaming the rules

The script runs full PPO (128 lines per iteration, 200 iterations, 3 seeds) from the same $\pi_{\text{ref}}$ and
measures on the 164 test prompts (4 lines per prompt) every 10 iterations. First, against the 4,000-pair reward model,
with β from 0 to 0.3.

### 🤔 Predict first
Gao et al. found that the KL penalty only makes the gold score stop earlier *along the same* gold–KL curve, like early
stopping, without lifting that curve. Is it the same here? That is: at the same KL divergence, does the β = 0.1 policy
have a higher gold score than the β = 0 policy?

In [ ]:
def rl_rows(**kw):
    return [x for x in RUNS["rl"] if all(x[k] == v for k, v in kw.items())]


BETAS = sorted({x["beta"] for x in rl_rows(algo="ppo", reward="rm", n=4000)})
colors = dict(zip(BETAS, ["black", ORANGE, GREEN, BLUE, GREY]))
markers = dict(zip(BETAS, ["x", "s", "^", "o", "D"]))
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
for beta in BETAS:
    rows = rl_rows(algo="ppo", reward="rm", n=4000, beta=beta)
    for row in rows:
        d = np.sqrt([max(e["kl"], 0) for e in row["log"]])
        axes[0].plot(d, [e["gold"] for e in row["log"]], color=colors[beta], marker=markers[beta], ms=3, lw=0.8,
                     label=f"β = {num(beta, 2)}" if row["seed"] == 0 else None)  # fmt: skip
    its = [e["it"] for e in rows[0]["log"]]
    v = seeds_of([r_["log"] for r_ in rows], "proxy")
    axes[1].plot(its, v.mean(0), color=colors[beta], marker=markers[beta], ms=3)
    axes[1].fill_between(its, v.min(0), v.max(0), color=colors[beta], alpha=0.15)
axes[0].set(xlabel="√KL(π‖π_ref)", ylabel="gold score", title="Gold against divergence (one curve per seed)")
axes[1].set(xlabel="PPO iteration", ylabel="proxy (normalized)", title="The reward model's score (3 seeds)")
axes[0].legend(frameon=False, fontsize=8)
fmt_axes(fig)
plt.show()
for beta in BETAS:
    rows = rl_rows(algo="ppo", reward="rm", n=4000, beta=beta)
    peak = [max(e["gold"] for e in r_["log"]) for r_ in rows]
    end = [r_["log"][-1] for r_ in rows]
    print(
        f"β = {num(beta, 2):>4}: highest gold {num(min(peak), 3)}–{num(max(peak), 3)}; at iteration 200 gold "
        f"{num(min(e['gold'] for e in end), 3)}–{num(max(e['gold'] for e in end), 3)}, KL "
        f"{num(min(e['kl'] for e in end), 1)}–{num(max(e['kl'] for e in end), 1)}"
    )
print("\nBest gold reached while the KL was still within a budget (range across the 3 seeds):")
for beta in BETAS:
    rows = rl_rows(algo="ppo", reward="rm", n=4000, beta=beta)
    best = {b: [max(e["gold"] for e in r_["log"] if e["kl"] <= b) for r_ in rows] for b in (2, 4, 16)}
    print(
        f"β = {num(beta, 2):>4}: " + " · ".join(f"KL ≤ {b}: {num(min(v), 3)}–{num(max(v), 3)}" for b, v in best.items())
    )

Here it does. For every KL budget, β = 0.1 reaches a higher gold than β = 0: at KL ≤ 4, 0.787–0.798 against
0.730–0.758, two ranges that don't overlap; at KL ≤ 2, β = 0.3 reaches 0.775–0.790 while β = 0 reaches only
0.725–0.737. Without a penalty, gold peaks (0.756–0.778) and then goes down, to 0.722–0.740 at iteration 200 with a KL
of 34.7–48.7, while the proxy (right figure) keeps rising or stays high: exactly the Goodhart shape that Gao et al.
measured. With a penalty, the gold–√KL curve is much steeper right from the start (left figure). A hypothesis, which
the notebook doesn't test: the KL penalty doesn't just stop the policy earlier, it also changes its *direction*,
because the policy has to keep most of $\pi_{\text{ref}}$'s distribution and only shifts probability where the
rewards differ a lot.

Gao et al. found otherwise, and they themselves note that their conclusion is sensitive to hyperparameters. A small
model with one set of PPO hyperparameters is not enough to settle the question. What is more certain is that β and
early stopping can't stand in for each other without measuring.

### A larger reward model, and RLOO


The figure below keeps β = 0.01 and varies the number of comparison pairs the reward model learns from (250 to
16,000), alongside RLOO (Ahmadian et al., 2024): REINFORCE without a value model, with the mean of the 3 other lines
for the same prompt as its baseline, and the KL computed over the whole line as in Lesson 26.

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4))
specs = [({"algo": "ppo", "n": 250, "beta": 0.01}, GREY, "v", "PPO, 250 pairs"),
         ({"algo": "ppo", "n": 1000, "beta": 0.01}, GREEN, "^", "PPO, 1,000 pairs"),
         ({"algo": "ppo", "n": 4000, "beta": 0.01}, BLUE, "o", "PPO, 4,000 pairs"),
         ({"algo": "ppo", "n": 16000, "beta": 0.01}, ORANGE, "s", "PPO, 16,000 pairs"),
         ({"algo": "rloo", "n": 4000, "beta": 0.01}, "black", "x", "RLOO, 4,000 pairs")]  # fmt: skip
for kw, color, marker, label in specs:
    rows = rl_rows(reward="rm", **kw)
    v = seeds_of([r_["log"] for r_ in rows], "gold")
    its = [e["it"] for e in rows[0]["log"]]
    ax.plot(its, v.mean(0), color=color, marker=marker, ms=4, label=label)
    ax.fill_between(its, v.min(0), v.max(0), color=color, alpha=0.15)
    print(
        f"{label:17}: gold at iteration 200 {num(v[:, -1].min(), 3)}–{num(v[:, -1].max(), 3)}, highest (seed mean) "
        f"{num(v.mean(0).max(), 3)} at iteration {int(its[v.mean(0).argmax()])}"
    )
ax.axhline(ev_sft["gold"], color=GREY, ls=":", lw=1)
ax.set(xlabel="iteration", ylabel="gold score", title="β = 0.01: by the reward model's number of comparison pairs")
ax.legend(frameon=False, fontsize=8)
fmt_axes(fig)
plt.show()

The more pairs the reward model learned from, the higher the gold: at iteration 200, 0.783–0.858 with 16,000 pairs
against 0.565–0.799 with 250 pairs. With 250 and 1,000 pairs, the seed mean peaks as early as iteration 20 (0.725 and
0.737): a weak reward model gets exploited early. Gao et al. found the same direction when they increased the data or
the size of the reward model. The ranges across seeds are wide, because each seed has its own set of pairs, its own
reward model and its own RL run.

RLOO reaches 0.696–0.772 at iteration 200, overlapping PPO with the same 4,000 pairs (0.717–0.814): within the
variation between seeds. Ahmadian et al. found RLOO on par with or better than PPO on large models; here we can only
say that it is not clearly worse, even though it needs no value model.

### Reading the lines that "game the rules"

The weights file includes the β = 0 PPO policy after 200 iterations (seed 0). The cell below has it write one line for
each test prompt.

In [ ]:
ppo_ship = frozen(unpack(GPT(), W["ppo"]))
ev_ppo, seq_ppo = evaluate(ppo_ship, {"proxy": R})
print("\n".join(decode(seq_ppo, 6)))
print(
    f"\ngold {num(ev_ppo['gold'], 3)} (SFT {num(ev_sft['gold'], 3)}) · proxy {num(ev_ppo['proxy'], 2)} · KL "
    f"{num(ev_ppo['kl'], 1)} nats · distinct-2 {num(ev_ppo['distinct2'], 3)}"
)
for name, v, v0 in zip(CHECKS, ev_ppo["checks"], ev_sft["checks"]):
    print(f"  {name:16} {pct(v):>6}  (SFT {pct(v0)})")
print("most frequent syllable at positions 2, 6, 8:", ", ".join(f"“{w}” {pct(s)}" for w, s in ev_ppo["top"]))
saved = RUNS["live"]["check"]["ppo"]
assert abs(ev_ppo["gold"] - saved["gold"]) < 0.01 and abs(ev_ppo["kl"] / saved["kl"] - 1) < 0.02, (
    "differs from the saved results"
)

Almost every line is the same line: "một tra … nước … tra … tra" (roughly “one tra … water … tra … tra”). The policy
has found exactly the blind spot of section 6. "Tra" has the ngang tone and sits at syllables 2, 6 and 8, so the
lines pass the five rules that the reward model sees (100.0%), while the huyền/ngang rule never passes (0.0%), because
syllables 6 and 8 are both "tra". The rhyme passes 18.3% of the time, on the prompts whose syllable 6 happens to rhyme
with "tra". The proxy is 1.79 (in units of the standard deviation of the score on SFT lines), but gold only goes from
0.679 to 0.740. With a KL of 43.4 nats and a distinct-2 of 0.018, the policy has almost forgotten every other line.

This is reward hacking you can read with your own eyes. In large models it is usually harder to see: long-winded
answers, telling the person asking what they want to hear (sycophancy), code that passes the tests without being
correct. MacDiarmid et al. (2025) found that a model that learned reward hacking in real programming environments
generalized it to other misaligned behaviors.

The cell below checks that the loaded weights really are the script's: the log-probabilities of 4 real lines of verse
(teacher forcing) under $\pi_{\text{ref}}$ and under the PPO policy, and the reward model's score.

In [ ]:
Xp = padded(X_test[:4])
with torch.no_grad():
    lp_s, lp_p, sc = seq_logp(sft, Xp), seq_logp(ppo_ship, Xp), rm(Xp)
for k in range(4):
    print(decode(Xp[k : k + 1])[0])
    print(
        f"    gold {num(gold(Xp[k : k + 1]).item(), 3)} · log π_ref {num(lp_s[k], 1)} · log π_PPO {num(lp_p[k], 1)} · "
        f"raw reward-model score {num(sc[k], 2)}"
    )
for name, v in (("sftLogp", lp_s), ("ppoLogp", lp_p), ("rm", sc)):
    assert np.allclose(v.numpy(), CHECK[name], atol=1e-3), "the weights differ from the script's"

All four lines are real verse and pass all 7 rules. Under the PPO policy, their log-probability is about 60 nats lower
than under $\pi_{\text{ref}}$: the policy no longer writes real verse. The reward model scores the last three lines
2.21–2.81 (the mean on SFT lines is 1.836), but scores the first −0.58, even though that line follows every rule. Its
error isn't just the two ignored rules.

### A rater who likes long lines, and reward-model ensembles

Two more experiments at β = 0.01, 3 seeds each:
- **A rater who likes long lines.** Singhal et al. (2023) found that most of RLHF's "progress" on several datasets
  came from longer answers. We give the rater a small bias: $u = g + 0.1\cdot(\text{number of syllables} - 8)$. The
  lục bát rules (gold) penalize every line that doesn't have 8 syllables.
- **Ensembles.** Coste et al. (2024) train several reward models and optimize against their *lowest* score
  (conservative) or their mean. We use 3 models, each trained on a bootstrap sample of the 4,000 pairs (Christiano et
  al., 2017).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
specs = [("rm", BLUE, "o", "ordinary reward model"), ("length", ORANGE, "s", "rater who likes long lines"),
         ("ens-min", GREEN, "^", "ensemble, min"), ("ens-mean", GREY, "D", "ensemble, mean")]  # fmt: skip
for spec, color, marker, label in specs:
    rows = rl_rows(algo="ppo", reward=spec, n=4000, beta=0.01)
    its = [e["it"] for e in rows[0]["log"]]
    for ax, key in zip(axes, ("len", "gold")):
        v = seeds_of([r_["log"] for r_ in rows], key)
        ax.plot(its, v.mean(0), color=color, marker=marker, ms=4, label=label)
        ax.fill_between(its, v.min(0), v.max(0), color=color, alpha=0.15)
    end = [r_["log"][-1] for r_ in rows]
    print(
        f"{label:26}: iteration 200 length {num(min(e['len'] for e in end), 1)}–{num(max(e['len'] for e in end), 1)} "
        f"syllables, gold {num(min(e['gold'] for e in end), 3)}–{num(max(e['gold'] for e in end), 3)}, "
        f"KL {num(min(e['kl'] for e in end), 1)}–{num(max(e['kl'] for e in end), 1)}"
    )
axes[0].set(xlabel="iteration", ylabel="mean number of syllables", title="Response length")
axes[1].set(xlabel="iteration", ylabel="gold score", title="Gold score")
axes[0].legend(frameon=False, fontsize=8)
fmt_axes(fig)
plt.show()
print(
    f"the rater who likes long lines flips {pct(RUNS['setup']['lengthFlip'])} of the labels "
    "relative to the ordinary rater"
)

The rater who likes long lines flips only 2.3% of the labels, but that is enough: the reward model learns that longer
is better, and PPO pulls every line up to 13.0 syllables, the maximum length the policy is allowed to write. Gold
falls to 0.502–0.637, below even SFT (0.679), because an eight-syllable line needs exactly 8 syllables. Singhal et al.
also found that RLHF with a reward based *only* on length reproduced most of the improvement over the SFT model.

An ensemble of three reward models (Coste et al., 2024), taking the min or the mean, gives a gold of 0.732–0.831 and
0.729–0.818 at iteration 200, against 0.717–0.814 for a single model: within the variation between seeds. Perhaps
because the three members learn from the same kind of data (bootstraps of the same 4,000 pairs, the same SFT body),
they share the same blind spot: no member sees the rhyme rule, and taking the min of three identical mistakes doesn't
fix that mistake. An ensemble helps when its members are wrong *in different ways*.

## 10 · DPO and IPO: learning directly from the pairs

Lesson 26 derived DPO: the optimal policy $\pi_\beta \propto \pi_{\text{ref}}\,e^{r/\beta}$ lets us write it the other
way round, $r = \beta \log(\pi_\beta/\pi_{\text{ref}}) + \text{constant}$, and then substitute that into the
Bradley–Terry loss:
$$\mathcal{L}_{\text{DPO}} = -\log\sigma(\beta\,h), \qquad
  h = \log\frac{\pi(y_w)}{\pi_{\text{ref}}(y_w)} - \log\frac{\pi(y_l)}{\pi_{\text{ref}}(y_l)}.$$
No reward model needed, no sampling during training. IPO (Azar et al., 2023) replaces $-\log\sigma$ with a square:
$\mathcal{L}_{\text{IPO}} = \big(h - \tfrac{1}{2\tau}\big)^2$.

The difference shows clearly when the rater **never changes its mind**. The cell below takes 3 responses, a uniform
$\pi_{\text{ref}}$, and three comparison pairs that always give the same result: response 1 ≻ response 2 ≻
response 3. We optimize the policy's logits by gradient descent.

In [ ]:
W_, L_ = np.array([0, 0, 1]), np.array([1, 2, 2])  # three pairs: 1 ≻ 2, 1 ≻ 3, 2 ≻ 3 (indexed from 0)


def tabular(kind, coef, steps, lr=0.5):
    th = np.zeros(3)  # logits; π_ref is uniform, so log π/π_ref = logit − a constant, and h doesn't depend on it
    for _ in range(steps):
        h = th[W_] - th[L_]
        e = -coef * sigmoid(-coef * h) if kind == "DPO" else 2 * (h - 1 / (2 * coef))  # ∂loss/∂h
        th -= lr * (np.bincount(W_, e, 3) - np.bincount(L_, e, 3)) / 3
    return np.exp(th) / np.exp(th).sum()


for kind, coef in (("DPO", 0.1), ("DPO", 1.0), ("IPO", 0.1), ("IPO", 1.0)):
    name = "β" if kind == "DPO" else "τ"
    rows = [tabular(kind, coef, s) for s in (100, 1000, 10000)]
    print(f"{kind} {name} = {num(coef, 1)}: " + " · ".join(f"{num(s, 0)} steps ({', '.join(num(v, 3) for v in q)})"
                                                        for s, q in zip((100, 1000, 10000), rows)))  # fmt: skip
c_ = 1 / (2 * 0.1)
closed = np.exp([4 * c_ / 3, 2 * c_ / 3, 0]) / np.exp([4 * c_ / 3, 2 * c_ / 3, 0]).sum()
print("IPO τ = 0.1, closed-form solution:", ", ".join(num(v, 3) for v in closed))
assert np.allclose(tabular("IPO", 0.1, 10000), closed, atol=1e-6)

With DPO, both values of β head to (1, 0, 0); β only changes the speed. The rater always picks response 1, so
$-\log\sigma(\beta h)$ only goes to 0 as $h \to \infty$, that is, as $\pi(y_2), \pi(y_3) \to 0$, for every β. β is
supposed to keep the policy close to $\pi_{\text{ref}}$, but here it appears only through the product $\beta h$, and
there is no finite optimum for it to hold the policy at. A reward model trained with the same Bradley–Terry loss also
pushes the score gap to infinity; Azar et al. note that in practice it is stopped early and regularized, so the reward
stays finite, and that is what DPO lacks.

The IPO loss is a square, with its minimum at $h = 1/(2\tau)$ for each pair, so the policy stops at a finite distance:
(0.964, 0.034, 0.001) with τ = 0.1, exactly the closed-form solution, and (0.448, 0.321, 0.230) with τ = 1. In IPO, τ
really controls the distance to $\pi_{\text{ref}}$. Xu et al. (2024) further show that DPO's solution set even
contains policies that pile probability onto responses that aren't in the data.

### DPO on lục bát verse

The cell below runs DPO with β = 0.1 for one pass over exactly the 1,000 pairs that the reward model of section 6
learned from (16 steps, batches of 64), and tracks the log-probabilities of the chosen and the rejected lines,
relative to where they started.

### 🤔 Predict first
DPO makes $h$ grow: the chosen line has to beat the rejected one. So does $\log\pi(y_w)$ of the chosen lines go up or
down? Is it the same on the *training* pairs and on the 500 *validation* pairs (never seen)?

In [ ]:
def ordered(a, b, y):
    yy = y.bool()[:, None]
    return torch.where(yy, a, b), torch.where(yy, b, a)


def dpo(win, lose, kind, beta, seed, epochs, lr=3e-4, batch=64, evals_per_epoch=2, held=None):
    torch.manual_seed(seed)
    pol = copy.deepcopy(sft)
    for p_ in pol.parameters():
        p_.requires_grad_(True)
    opt = torch.optim.Adam(pol.parameters(), lr)
    g = torch.Generator().manual_seed(60_000 + seed)
    with torch.no_grad():
        rw, rl, hw, hl = seq_logp(sft, win), seq_logp(sft, lose), seq_logp(sft, held[0]), seq_logp(sft, held[1])
    steps_per_epoch = -(-len(win) // batch)
    marks = {round(steps_per_epoch * k / evals_per_epoch) for k in range(1, evals_per_epoch * epochs + 1)}
    log, step = [], 0

    def record():
        with torch.no_grad():
            dw, dl = seq_logp(pol, held[0]) - hw, seq_logp(pol, held[1]) - hl
            log.append({"step": step, "chosen": (seq_logp(pol, win) - rw).mean().item(),
                        "rejected": (seq_logp(pol, lose) - rl).mean().item(), "held_chosen": dw.mean().item(),
                        "held_rejected": dl.mean().item(), "held_acc": (dw > dl).float().mean().item()})  # fmt: skip

    record()
    for _ in range(epochs):
        for j in torch.randperm(len(win), generator=g).split(batch):
            h = (seq_logp(pol, win[j]) - rw[j]) - (seq_logp(pol, lose[j]) - rl[j])
            loss = -F.logsigmoid(beta * h).mean() if kind == "dpo" else ((h - 1 / (2 * beta)) ** 2).mean()
            opt.zero_grad()
            loss.backward()
            opt.step()
            step += 1
            if step in marks:
                record()
    return pol, log


win, lose = ordered(a_tr, b_tr, y_tr)
_, dlog = dpo(win, lose, "dpo", cfg["dpoBeta"], 0, cfg["dpoEpochs"], evals_per_epoch=cfg["dpoEvals"],
              held=ordered(*pairs["val"]))  # fmt: skip
print("step     Δ log π: chosen   rejected   | validation: chosen   rejected   h > 0")
for e in dlog:
    print(
        f"{e['step']:4}   {num(e['chosen'], 3):>17}   {num(e['rejected'], 3):>8}   | {num(e['held_chosen'], 3):>18}"
        f"   {num(e['held_rejected'], 3):>8}   {pct(e['held_acc'])}"
    )
assert abs(dlog[-1]["chosen"] - RUNS["live"]["dpo"][-1]["chosen"]) < 0.02, "differs from the saved results"

On the training pairs, the log π of the chosen line rises only slightly (0.085 nats after 16 steps) while that of the
rejected line drops sharply (−0.469): nearly all of the increase in $h$ comes from pushing the rejected line down. On
the validation pairs, *both* drop (−0.070 and −0.174): chosen lines the model hasn't seen lose probability too. DPO
only constrains the difference $h$, not each of its two terms. Pal et al. (2024) showed that DPO can lower the
probability of the chosen response, especially when the two responses differ in only a few tokens. Razin et al.
(2024) call this *likelihood displacement* and find it strongest when the representations of the chosen and the
rejected response are similar, like two eight-syllable lines for the same prompt here. After one pass, the policy
ranks 54.8% of the validation pairs correctly, only slightly better than guessing.

The script runs DPO (β 0.03 / 0.1 / 0.3) and IPO (τ 0.01 / 0.03 / 0.1) for 6 passes over 4,000 pairs, 3 seeds,
measuring gold and KL on the test prompts twice per pass. The left figure puts them next to PPO with β = 0.1 and
best-of-N on the same gold–√KL axes. The right figure plots the log-probabilities of the chosen and the rejected lines
(training pairs) over the passes of DPO with β = 0.1.

In [ ]:
def dpo_rows(**kw):
    return [x for x in RUNS["dpo"] if all(x[k] == v for k, v in kw.items())]


fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
specs = [({"kind": "dpo", "beta": 0.1, "n": 4000}, BLUE, "o", "DPO β = 0.1"),
         ({"kind": "dpo", "beta": 0.03, "n": 4000}, GREEN, "^", "DPO β = 0.03"),
         ({"kind": "ipo", "beta": 0.1, "n": 4000}, ORANGE, "s", "IPO τ = 0.1")]  # fmt: skip
for kw, color, marker, label in specs:
    for row in dpo_rows(**kw):
        axes[0].plot(np.sqrt([max(e["kl"], 0) for e in row["log"]]), [e["gold"] for e in row["log"]], color=color,
                     marker=marker, ms=3, lw=0.8, label=label if row["seed"] == 0 else None)  # fmt: skip
for row in rl_rows(algo="ppo", reward="rm", n=4000, beta=0.1):
    axes[0].plot(np.sqrt([max(e["kl"], 0) for e in row["log"]]), [e["gold"] for e in row["log"]], color="black",
                 marker="x", ms=3, lw=0.8, label="PPO β = 0.1" if row["seed"] == 0 else None)  # fmt: skip
bon = [b["curves"]["rm4000"] for b in RUNS["bon"]]
axes[0].plot(np.sqrt([e["klBound"] for e in bon[0]]), seeds_of(bon, "gold").mean(0), color=GREY, ls="--",
             label="best-of-N (seed mean)")  # fmt: skip
axes[0].set(xlabel="√KL(π‖π_ref)", ylabel="gold score", title="The same 4,000 pairs, three ways to use them")
axes[0].legend(frameon=False, fontsize=7)
for row in dpo_rows(kind="dpo", beta=0.1, n=4000):
    ep = [e["epoch"] for e in row["log"]]
    axes[1].plot(ep, [e["chosen"] for e in row["log"]], color=BLUE, marker="o", ms=3,
                 label="chosen line" if row["seed"] == 0 else None)  # fmt: skip
    axes[1].plot(ep, [e["rejected"] for e in row["log"]], color=ORANGE, marker="s", ms=3, ls="--",
                 label="rejected line" if row["seed"] == 0 else None)  # fmt: skip
axes[1].axhline(0, color=GREY, lw=0.8)
axes[1].set(xlabel="pass", ylabel="Δ log π relative to π_ref (nats)", title="DPO β = 0.1 on the training pairs")
axes[1].legend(frameon=False, fontsize=8)
fmt_axes(fig)
plt.show()


def span(end, key, nd):
    return f"{num(min(e[key] for e in end), nd)}–{num(max(e[key] for e in end), nd)}"


for kind, beta, n in sorted({(x["kind"], x["beta"], x["n"]) for x in RUNS["dpo"]}, key=lambda k: (k[0], k[2], k[1])):
    end = [r_["log"][-1] for r_ in dpo_rows(kind=kind, beta=beta, n=n)]
    best = [max(e["gold"] for e in r_["log"]) for r_ in dpo_rows(kind=kind, beta=beta, n=n)]
    print(f"{kind.upper()} {'β' if kind == 'dpo' else 'τ'} = {num(beta, 2)}, {num(n, 0)} pairs: highest gold "
          f"{num(min(best), 3)}–{num(max(best), 3)}; at pass 6: gold {span(end, 'gold', 3)}, "
          f"KL {span(end, 'kl', 1)}")  # fmt: skip
    print(f"    pass 6: Δ log π chosen {span(end, 'chosen', 1)}, rejected {span(end, 'rejected', 1)}; "
          f"validation pairs ranked correctly {pct(min(e['held_acc'] for e in end))}–"
          f"{pct(max(e['held_acc'] for e in end))}")  # fmt: skip

Right figure: likelihood displacement goes on through all 6 passes. With β = 0.1, the chosen line loses 6.2–7.7 nats,
on the training pairs themselves, while the rejected line loses 16.2–20.6. The smaller β, the deeper both fall and
the larger the KL. That probability goes to lines that appear in no pair, which no one grades. With β = 0.03 and with
IPO τ = 0.01, gold at pass 6 drops below SFT: there is no reward model to game, yet DPO still over-optimizes. Rafailov
et al. (2024) measure the same phenomenon in direct alignment algorithms.

Left figure: with the same 4,000 pairs, PPO with β = 0.1 reaches 0.787–0.798 at KL ≤ 4. The highest gold of DPO and
IPO is lower: 0.742–0.749 with DPO β = 0.3 (KL 2.4–3.7 at pass 6), 0.739–0.747 with IPO τ = 0.1, and 0.721–0.748 with
DPO β = 0.1, which then drops to 0.624–0.714 at pass 6. After 6 passes, DPO with β = 0.1 ranks 54.7–63.1% of the
validation pairs correctly, worse than the reward model learned from the same 4,000 pairs (67.5–68.2% on the test
pairs). DPO learns from pairs of lines written by $\pi_{\text{ref}}$; once the policy has moved far from
$\pi_{\text{ref}}$, it has no signal about the very lines it is writing. PPO samples new lines at each iteration and
asks the reward model, and the reward model generalizes from 4,000 pairs to every line. Xu et al. (2024) also found
that well-tuned PPO beats DPO on many tasks; iterative DPO (sampling anew from the policy, labeling again) narrows the
gap, but here we only run offline DPO. More pairs help DPO too: with β = 0.1, the highest gold is 0.699–0.717 with
1,000 pairs and 0.755–0.766 with 16,000 pairs.

## 11 · Verifiable rewards (RLVR)

If quality *can* be written as a program, like the answer to a math problem or the tests of a piece of code, we skip
the reward model and optimize directly against that program. That is Tülu 3's RLVR, and the way DeepSeek-R1 learned to
reason. Here the program is gold itself. The figure below compares PPO against gold (β = 0 and 0.01) with PPO against
the 4,000-pair reward model: the gold score, the fraction of lines whose syllable 6 rhymes with the prompt (the hardest
check), and distinct-2, a measure that gold doesn't grade.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3.4))
specs = [({"reward": "gold", "beta": 0.0}, "black", "x", "gold, β = 0"),
         ({"reward": "gold", "beta": 0.01}, GREEN, "^", "gold, β = 0.01"),
         ({"reward": "rm", "beta": 0.01}, BLUE, "o", "reward model, β = 0.01")]  # fmt: skip
for kw, color, marker, label in specs:
    rows = rl_rows(algo="ppo", n=4000, **kw)
    its = [e["it"] for e in rows[0]["log"]]
    for ax, key in zip(axes, ("gold", "rhyme", "distinct2")):
        v = np.array([[e["checks"][5] if key == "rhyme" else e[key] for e in r_["log"]] for r_ in rows])
        ax.plot(its, v.mean(0), color=color, marker=marker, ms=3, label=label)
        ax.fill_between(its, v.min(0), v.max(0), color=color, alpha=0.15)
    end = [r_["log"][-1] for r_ in rows]
    print(
        f"{label:22}: iteration 200 gold {num(min(e['gold'] for e in end), 3)}–{num(max(e['gold'] for e in end), 3)}, "
        f"KL {num(min(e['kl'] for e in end), 1)}–{num(max(e['kl'] for e in end), 1)}, distinct-2 "
        f"{num(min(e['distinct2'] for e in end), 2)}–{num(max(e['distinct2'] for e in end), 2)}, rhyme "
        f"{pct(min(e['checks'][5] for e in end))}–{pct(max(e['checks'][5] for e in end))}"
    )
for ax, t_ in zip(axes, ("Gold score", "Syllable 6 rhymes", "Distinct-2")):
    ax.set(xlabel="iteration", title=t_)
axes[0].legend(frameon=False, fontsize=7)
fmt_axes(fig)
plt.show()
row = rl_rows(algo="ppo", reward="gold", n=4000, beta=0.0, seed=0)[0]
print("\n".join(row["lines"]["200"][:4]))
coda_i = (CODA[X_test[:, 5]] == codas.index("i")).float().mean().item()
print(f"test prompts whose syllable 6 ends in the coda i (like “lời”, “nơi”): {pct(coda_i)}")

Against gold itself there is nothing to game: gold rises to 0.888–0.893 with β = 0, higher than with any reward
model. But read the lines: "có nơi lời có lời lời lại nơi" (word for word, roughly “have place words have words words
again place”). The lines follow the rules but make no sense, only repeating a few words (distinct-2 0.16–0.29). Gold
grades the rules, not the meaning, so RL optimizes exactly what is graded, nothing more. With β = 0.01, the lines are
far more varied (distinct-2 0.69–0.91) and gold reaches 0.795–0.890.

The syllable-6 rhyme stops at exactly 26.8% in all three seeds, exactly the fraction of test prompts whose syllable 6
ends in the coda i (26.8%). The policy has learned a trick: put "lời" (“words”) at syllable 6 and "nơi" (“place”) at
syllable 8. Under the loose-rhyme rule, all syllables ending in i rhyme with each other, so the trick wins on every
prompt ending in i, and "lời" (huyền) with "nơi" (ngang) also passes the syllable 6–8 rule. On the other prompts, the
policy finds no way to rhyme, because $\pi_{\text{ref}}$ rarely rhymes (12.8%) and each rhyme needs different words.
RL amplifies what $\pi_{\text{ref}}$ already did now and then; it has a hard time finding what $\pi_{\text{ref}}$
almost never does. Yue et al. (2025) ask the same question of RLVR on math: for large k, the pass@k of the base model
equals or beats that of the model after RL; the question is still being debated. The reward model (β = 0.01) reaches a
rhyme rate of only 11.9–14.8%, close to the SFT level: it doesn't see the rhyme rule, so it doesn't reward it.

In practice, verifiable rewards are used where right and wrong are clear-cut: Tülu 3 runs RLVR after DPO, only on
problems whose answers can be checked; DeepSeek-R1 uses rule-based rewards for math and code, and reward models for
the rest.

## 12 · Try changing things

### The tail of the error

Section 7 compared a normal error with a Student-t error with ν = 1.5. The cell below tries tails in between: the
larger ν, the closer the Student-t distribution is to the normal distribution. You can edit the list and run it again.

In [ ]:
for nu in (1.5, 3, 10):
    r_try = g_tab + 0.6 * np.random.default_rng(1).standard_t(nu, K)
    gb_ = np.array([best_of(r_try, N) @ g_tab for N in NS_TAB])
    print(
        f"ν = {num(nu, 1):>4}: best-of-N highest gold {num(gb_.max(), 2)} at N = {num(NS_TAB[gb_.argmax()], 0)}, "
        f"at N = 4,096: {num(gb_[-1], 2)}"
    )

### A sharper or a noisier rater

Changing $s$ changes both the accuracy ceiling and how often two raters agree. The cell below computes both on the
1,000 training pairs.

In [ ]:
for s_ in (3, 10, 30, 100):
    p_ = torch.sigmoid(s_ * (ga - gb))
    print(
        f"s = {s_:3}: two raters agree {pct((p_**2 + (1 - p_) ** 2).mean())}, "
        f"ceiling {pct(torch.maximum(p_, 1 - p_).mean())}"
    )

### KL in the bandit problem with other values of β

The cell below computes section 8's two targets, $\pi^*$ (KL in the reward) and $\pi_{\text{rev}}$ ($k_3$ in the
loss), from their closed-form solutions for a few other values of β, with no training. To see the learned policy,
change `beta_b` in section 8 and rerun the cells from there on.

In [ ]:
for beta_try in (0.03, 0.1, 0.3):
    s_star, s_rev = torch.softmax(lp_ref_b + r_band / beta_try, 0), pi_rev(beta_try)
    print(
        f"β = {num(beta_try, 2)}: TV(π*, π_rev) {num(tv(s_star, s_rev), 3)} · largest probability: "
        f"π* {pct(s_star.max())}, π_rev {pct(s_rev.max())}"
    )
    if beta_try == beta_b:
        assert abs(tv(s_star, s_rev) - tv(star, rev)) < 1e-9

## 13 · Limitations

- **Gold is a 7-rule program, not human taste.** That is what lets us measure Goodhart exactly, but the reward model's
  blind spot here is simple (two ignored rules). Real reward models go wrong in ways that are more varied and harder
  to see.
- **The simulated rater has one parameter**, $s$, plus a variant that likes long lines. Real people get tired, are
  inconsistent, have their own views, and are sometimes fooled by fluent answers.
- **A very small model:** 254,016 parameters and 1,300 couplets, so the lines have no meaning and we can only measure
  the rules. In large models, the reward model starts from a model that already understands language, and it
  generalizes much better.
- **One set of PPO hyperparameters**, chosen in advance by the clip fraction, not tuned. The conclusion "the KL penalty
  lifts the gold–KL curve" might change with another set.
- **DPO is only run offline.** Iterative DPO and other variants (KTO, SimPO, ORPO) aren't covered here; GRPO is in
  Lesson 26.
- **3 seeds.** Many ranges overlap, and those conclusions are stated as being within the variation between seeds.

## Summary
- **RLHF** has three steps: SFT, then a Bradley–Terry reward model learned from comparison pairs, then RL (PPO) with a
  KL penalty toward $\pi_{\text{ref}}$. Bradley–Terry determines the reward only up to a constant for each prompt.
- **Comparison labels are inherently noisy:** on the script's 2,000 test pairs, the accuracy ceiling is 78.0%, and two
  raters agree only 73.2% of the time. The 4,000-pair reward model predicts 67.5–68.2% of the labels of those pairs
  correctly, but almost ignores the rhyme rule and the huyền/ngang rule.
- **PPO without a penalty finds that blind spot:** "một tra … tra", a proxy of 1.79, gold only 0.740, a KL of 43.4
  nats. Gold peaks and then falls while the proxy keeps rising.
- **A heavy-tailed error** makes the gold of best-of-N and of $\pi_\beta$ fall below even $\pi_{\text{ref}}$
  (catastrophic Goodhart). On this problem, the KL penalty β = 0.1 gives a higher gold than β = 0 at the same KL
  budget.
- **Where you put the KL matters:** in the reward, the policy goes exactly to $\pi^*$; $k_1$ in the loss penalizes
  nothing at all; $k_3$ in the loss penalizes the KL in the reverse direction.
- **DPO** drops the reward model, but has no finite optimum when preferences are absolute (IPO does). On the verse, it
  lowers even the probability of the chosen line (likelihood displacement), and run offline it loses to PPO at the
  same KL.
- **A verifiable reward** can't be gamed, but RL optimizes exactly what is graded: lines that follow the rules, make no
  sense, and rhyme only where there is already a trick that $\pi_{\text{ref}}$ sometimes used.

## Review questions
1. The 4,000-pair reward model predicts only 67.5–68.2% of the labels correctly. Why is that number not as bad as it
   sounds?
2. Add 5 to the score of every response to one prompt. How does the Bradley–Terry loss change? Are PPO and RLOO
   affected?
3. In the heavy-tailed best-of-N figure of section 7, why does gold fall as N grows, even though the KL of best-of-N is
   at most $\log N - (N-1)/N$?
4. An algorithm adds $\beta\,k_3$ to the loss. Toward which distribution does it steer the policy? How does that
   differ from RLHF's $\pi^*$?
5. Why does DPO with absolute preferences take the policy to (1, 0, 0) for every β, while IPO doesn't?
6. The β = 0 PPO policy writes "một tra … tra". Explain why this line passes five rules but not the huyền/ngang rule,
   then propose two ways to prevent it.

<details><summary>Hints and answers</summary>

1. The labels are noisy: of two lines that differ by one rule, the better one is chosen only with probability 0.807,
   and when two lines tie, the label is a coin flip. The ceiling for any model is 78.3% on the notebook's 1,000 test
   pairs (the model that knows all the rules gets 79.4% on this sample) and 78.0% on the script's 2,000 test pairs, and
   two raters agree only 73.2% of the time. The 67.5–68.2% is measured on those 2,000 pairs: read it against the 78.0%
   ceiling, not against 100%.
2. The loss doesn't change, because it only uses the difference between the scores of two responses to the same
   prompt. RLOO subtracts the mean of the other responses to the same prompt, so the constant is removed entirely. PPO
   subtracts the value $V(s_t)$; once the value model has learned that constant, the advantage doesn't change. That is
   why the notebook normalizes the scores globally, with no need to normalize per prompt.
3. With a heavy tail, the winner among N lines is the line with the largest error, not the line with the highest gold,
   and the larger N, the surer it is to pick a huge error. A small KL only says that the policy is close to
   $\pi_{\text{ref}}$; a small amount of probability placed right on those lines is enough to push the proxy up high
   (Kwa et al., 2024).
4. The expected gradient is $\nabla\,\mathrm{KL}(\pi_{\text{ref}}\,\|\,\pi)$, so the policy heads toward
   $\pi_{\text{rev}} = \beta\,\pi_{\text{ref}}/(\lambda - r)$, not $\pi^* \propto \pi_{\text{ref}}\,e^{r/\beta}$:
   the reward enters through a fraction instead of an exponential. In the bandit problem, the TV distance between the
   two distributions is 0.520.
5. With DPO, the loss of each pair keeps decreasing as $h$ grows, so the optimum lies at infinity for every β. IPO
   penalizes the squared deviation of $h$ from $1/(2\tau)$, so each pair has a finite optimal $h$.
6. "Tra" has the ngang tone and sits at syllables 2, 6 and 8, so the line passes the length and tone rules that the
   reward model rewards. Syllables 6 and 8 both have the ngang tone, so the huyền/ngang rule always fails, but the
   reward model hardly sees that rule (coefficient −0.02). Ways to prevent it: a KL penalty (β = 0.1 keeps the KL at
   1.9–3.7); collecting more labels on the policy's own lines and retraining the reward model, as InstructGPT and
   Llama 3 do over several rounds; adding a rule-based checker for the part that can be verified.
</details>

## Further reading
- Christiano et al. (2017), *Deep reinforcement learning from human preferences*:
  [arXiv:1706.03741](https://arxiv.org/abs/1706.03741).
- Ziegler et al. (2019), *Fine-Tuning Language Models from Human Preferences*:
  [arXiv:1909.08593](https://arxiv.org/abs/1909.08593).
- Stiennon et al. (2020), *Learning to summarize from human feedback*:
  [arXiv:2009.01325](https://arxiv.org/abs/2009.01325).
- Ouyang et al. (2022), *Training language models to follow instructions with human feedback* (InstructGPT):
  [arXiv:2203.02155](https://arxiv.org/abs/2203.02155).
- Bai et al. (2022), *Training a Helpful and Harmless Assistant with RLHF*:
  [arXiv:2204.05862](https://arxiv.org/abs/2204.05862); *Constitutional AI*:
  [arXiv:2212.08073](https://arxiv.org/abs/2212.08073).
- Schulman et al. (2017), *Proximal Policy Optimization Algorithms*:
  [arXiv:1707.06347](https://arxiv.org/abs/1707.06347); GAE (2016): [arXiv:1506.02438](https://arxiv.org/abs/1506.02438).
- Huang et al. (2024), *The N+ Implementation Details of RLHF with PPO*:
  [arXiv:2403.17031](https://arxiv.org/abs/2403.17031).
- Gao, Schulman & Hilton (2022), *Scaling Laws for Reward Model Overoptimization*:
  [arXiv:2210.10760](https://arxiv.org/abs/2210.10760).
- Kwa, Thomas & Garriga-Alonso (2024), *Catastrophic Goodhart: regularizing RLHF with KL divergence does not mitigate
  heavy-tailed reward misspecification*: [arXiv:2407.14503](https://arxiv.org/abs/2407.14503).
- Coste et al. (2024), *Reward Model Ensembles Help Mitigate Overoptimization*:
  [arXiv:2310.02743](https://arxiv.org/abs/2310.02743).
- Singhal et al. (2023), *A Long Way to Go: Investigating Length Correlations in RLHF*:
  [arXiv:2310.03716](https://arxiv.org/abs/2310.03716).
- Rafailov et al. (2023), *Direct Preference Optimization*: [arXiv:2305.18290](https://arxiv.org/abs/2305.18290).
- Azar et al. (2023), *A General Theoretical Paradigm to Understand Learning from Human Preferences* (IPO):
  [arXiv:2310.12036](https://arxiv.org/abs/2310.12036).
- Pal et al. (2024), *Smaug: Fixing Failure Modes of Preference Optimisation with DPO-Positive*:
  [arXiv:2402.13228](https://arxiv.org/abs/2402.13228).
- Razin et al. (2024), *Unintentional Unalignment: Likelihood Displacement in DPO*:
  [arXiv:2410.08847](https://arxiv.org/abs/2410.08847).
- Rafailov et al. (2024), *Scaling Laws for Reward Model Overoptimization in Direct Alignment Algorithms*:
  [arXiv:2406.02900](https://arxiv.org/abs/2406.02900).
- Xu et al. (2024), *Is DPO Superior to PPO for LLM Alignment?*: [arXiv:2404.10719](https://arxiv.org/abs/2404.10719).
- Ahmadian et al. (2024), *Back to Basics: Revisiting REINFORCE Style Optimization for Learning from Human Feedback in
  LLMs* (RLOO): [arXiv:2402.14740](https://arxiv.org/abs/2402.14740).
- Tang & Munos (2025), *On a few pitfalls in KL divergence gradient estimation for RL*:
  [arXiv:2506.09477](https://arxiv.org/abs/2506.09477).
- Grattafiori et al. (2024), *The Llama 3 Herd of Models*: [arXiv:2407.21783](https://arxiv.org/abs/2407.21783).
- Lambert et al. (2024), *Tülu 3*: [arXiv:2411.15124](https://arxiv.org/abs/2411.15124).
- Shao et al. (2024), *DeepSeekMath* (GRPO): [arXiv:2402.03300](https://arxiv.org/abs/2402.03300).
- DeepSeek-AI (2025), *DeepSeek-R1*: [arXiv:2501.12948](https://arxiv.org/abs/2501.12948).
- Yue et al. (2025), *Does Reinforcement Learning Really Incentivize Reasoning Capacity in LLMs Beyond the Base
  Model?*: [arXiv:2504.13837](https://arxiv.org/abs/2504.13837).
- MacDiarmid et al. (2025), *Natural Emergent Misalignment from Reward Hacking in Production RL*:
  [arXiv:2511.18397](https://arxiv.org/abs/2511.18397).
- Nathan Lambert, *Reinforcement Learning from Human Feedback* (open book): [rlhfbook.com](https://rlhfbook.com/).

**Back to:** [Lesson 26 · From language model to assistant](https://d3qr1cjln6qqtk.cloudfront.net/en/lessons/hau-huan-luyen/).

In [ ]:
import platform

import matplotlib

print(
    f"Python {platform.python_version()} · numpy {np.__version__} · torch {torch.__version__} · "
    f"matplotlib {matplotlib.__version__}"
)